# Care Compliance Copilot
## MVPロジック検証ノート



### 1. Google Driveとの接続


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### 2. ライブラリ読み込み・疑似CSVの読み込み


In [2]:
import pandas as pd
from pathlib import Path
import json

BASE_DIR = Path(
    "/content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ"
)

users = pd.read_csv(BASE_DIR / "users.csv")
care_plans = pd.read_csv(BASE_DIR / "care_plans.csv")
daily_records = pd.read_csv(BASE_DIR / "daily_records.csv")
monitoring_records = pd.read_csv(BASE_DIR / "monitoring_records.csv")
document_status = pd.read_csv(BASE_DIR / "document_status.csv")

print("CSVの読み込みが完了しました")
print("users:", users.shape)
print("care_plans:", care_plans.shape)
print("daily_records:", daily_records.shape)
print("monitoring_records:", monitoring_records.shape)
print("document_status:", document_status.shape)

CSVの読み込みが完了しました
users: (5, 6)
care_plans: (5, 9)
daily_records: (15, 15)
monitoring_records: (5, 18)
document_status: (10, 9)


### 3. 共通関数

In [3]:

def clean_text(value) -> str:
    """
    NaNやNoneを空文字へ変換し、前後の空白を除去する。
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def split_items(value) -> list[str]:
    """
    カンマ区切り、読点区切りの文字列をリストへ変換する。
    空欄は空リストにする。
    """
    text = clean_text(value)

    if not text:
        return []

    return [
        item.strip()
        for item in text.replace("、", ",").split(",")
        if item.strip()
    ]


def to_bool(value) -> bool:
    """
    CSVから読み込んだ True / False を安全にboolへ変換する。
    """
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() == "true"

### 4. データ前処理


In [4]:

# 今回のMVPでは、2026年6月10日時点の書類・記録状況を確認する
CHECK_DATE = pd.Timestamp("2026-06-10")

# --------------------------------------------------
# care_plans.csv の日付変換
# --------------------------------------------------

care_plans["valid_from"] = pd.to_datetime(
    care_plans["valid_from"],
    errors="coerce"
)

care_plans["valid_to"] = pd.to_datetime(
    care_plans["valid_to"],
    errors="coerce"
)

# --------------------------------------------------
# daily_records.csv の日付変換
# --------------------------------------------------

daily_records["record_date"] = pd.to_datetime(
    daily_records["record_date"],
    errors="coerce"
)

# --------------------------------------------------
# monitoring_records.csv の日付変換
# --------------------------------------------------

monitoring_records["record_date"] = pd.to_datetime(
    monitoring_records["record_date"],
    errors="coerce"
)

monitoring_records["interview_date"] = pd.to_datetime(
    monitoring_records["interview_date"],
    errors="coerce"
)

# --------------------------------------------------
# document_status.csv の日付変換
# --------------------------------------------------

document_status["valid_from"] = pd.to_datetime(
    document_status["valid_from"],
    errors="coerce"
)

document_status["valid_to"] = pd.to_datetime(
    document_status["valid_to"],
    errors="coerce"
)

# --------------------------------------------------
# True / False の表記ゆれ対策
# --------------------------------------------------

document_status["required"] = document_status["required"].apply(to_bool)
document_status["file_exists"] = document_status["file_exists"].apply(to_bool)
document_status["approved"] = document_status["approved"].apply(to_bool)

daily_records["approved"] = daily_records["approved"].apply(to_bool)

# --------------------------------------------------
# 特記事項の文字数を計算
# --------------------------------------------------

daily_records["notes_length"] = (
    daily_records["special_notes"]
    .fillna("")
    .astype(str)
    .str.len()
)

# --------------------------------------------------
# 前処理完了確認
# --------------------------------------------------

print("✅ データ前処理が完了しました")
print("CHECK_DATE:", CHECK_DATE.date())

print("care_plans:", care_plans.shape)
print("daily_records:", daily_records.shape)
print("monitoring_records:", monitoring_records.shape)
print("document_status:", document_status.shape)

✅ データ前処理が完了しました
CHECK_DATE: 2026-06-10
care_plans: (5, 9)
daily_records: (15, 16)
monitoring_records: (5, 18)
document_status: (10, 9)


### 5. データ確認

In [5]:
print("=== データ確認 ===")
display(users.head())
display(care_plans.head())
display(daily_records.head())
display(monitoring_records.head())
display(document_status.head())

=== データ確認 ===


,user_id,user_name,service_type,care_level,start_date,status
0,U001,利用者A,訪問介護,要介護2,2026-04-01,active
1,U002,利用者B,訪問介護,要介護3,2026-03-15,active
2,U003,利用者C,訪問介護,要介護2,2026-05-01,active
3,U004,利用者D,訪問介護,要介護4,2026-02-10,active
4,U005,利用者E,訪問介護,要介護1,2026-05-20,active


,plan_id,user_id,valid_from,valid_to,goal,service_content,precautions,approved_by,approved_at
0,P001,U001,2026-06-01,2026-11-30,安全に居室内を移動し、排泄動作を継続する,歩行見守り、声かけ、排泄介助,立ち上がり時のふらつきに注意,管理者A,2026-05-28
1,P002,U002,2026-04-01,2026-09-30,清潔保持と服薬管理を継続する,更衣介助、服薬確認、体調観察,血圧変動に注意,管理者A,2026-03-28
2,P003,U003,2026-03-01,2026-08-31,安全に室内歩行を継続する,歩行訓練、歩行見守り、必要時の声かけ,下肢筋力低下と転倒に注意,管理者B,2026-02-26
3,P004,U004,2026-05-01,2026-10-31,食事摂取量を維持し、体調を安定させる,食事見守り、水分摂取の声かけ、体調観察,食欲低下、発熱、脱水兆候に注意,管理者A,2026-04-29
4,P005,U005,2026-01-01,2026-06-30,自立した生活動作を維持する,生活援助、掃除、買い物支援、体調確認,疲労時は休憩を促す,管理者B,2025-12-26


,record_id,user_id,record_date,service_items,checklist_completed,special_notes,staff_name,approved,approved_at,user_quote,observation,numeric_data,follow_up,ai_draft,ai_draft_status,notes_length
0,R001,U001,2026-06-06,"歩行見守り,排泄介助",True,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,職員A,True,2026-06-06 18:00,今日は大丈夫,立ち上がり時に軽度ふらつきあり。歩行器使用。転倒なし。,トイレ移動1回,次回も立ち上がり時のふらつきを確認する,NaN,not_generated,69
1,R002,U001,2026-06-08,"歩行見守り,排泄介助",True,トイレ移動時に歩行器を使用。声かけに応じてゆっくり移動され、転倒なく終了した。本人より体調不...,職員B,True,2026-06-08 18:10,体調は悪くない,歩行器使用。声かけに応じて移動。転倒なし。,トイレ移動1回,現行支援を継続する,NaN,not_generated,56
2,R003,U001,2026-06-10,"歩行見守り,排泄介助",True,居室内の移動は歩行器で実施。立位保持は安定しており、排泄動作も普段通りであった。継続して安全...,職員A,True,2026-06-10 17:55,いつも通りです,立位保持は安定。排泄動作は普段通り。,居室内移動1回,安全確認を継続する,NaN,not_generated,52
3,R004,U002,2026-06-06,"服薬確認,体調観察",True,朝食後の服薬を確認した。血圧は普段と大きな変化なく、倦怠感や頭痛の訴えはなかった。更衣時は一...,職員C,True,2026-06-06 12:30,頭痛はないです,倦怠感なし。頭痛なし。更衣時に一部介助。,血圧128/74mmHg,服薬確認と体調観察を継続する,NaN,not_generated,54
4,R005,U002,2026-06-09,"服薬確認,更衣介助",True,服薬忘れがないことを確認した。上衣の着脱時に介助を実施。本人より体調は変わらないとの発言があった。,職員C,True,2026-06-09 12:20,体調は変わらない,服薬忘れなし。上衣着脱時に介助。,NaN,計画書の更新状況を確認する,NaN,not_generated,49


,monitoring_id,user_id,record_date,evaluation,status_change,next_action,approved_by,approved,interview_date,interview_method,attendees,information_sources,user_quotes,objective_facts,goal_evaluation,future_action,ai_draft,ai_draft_status
0,M001,U001,2026-06-10,歩行器を使用した居室内移動と排泄動作は概ね安定している。立ち上がり時のふらつきに引き続き注意する。,大きな変化なし,現行支援を継続する,サ責A,True,2026-06-10,訪問,本人,"本人,日次記録","今日は大丈夫,いつも通りです",歩行器を使用。立ち上がり時に軽度ふらつきあり。転倒なし。,概ね計画どおり。安全確認は継続が必要。,現行支援を継続する,NaN,not_generated
1,M002,U002,2026-05-25,服薬管理と更衣介助を継続している。体調は概ね安定している。,大きな変化なし,計画書の更新要否を確認する,サ責A,True,2026-05-25,訪問,本人,"本人,日次記録",頭痛はないです,服薬忘れなし。更衣時に一部介助。血圧に大きな変化なし。,支援は概ね継続できているが、計画書の期限確認が必要。,計画書を更新し、現行支援の継続要否を確認する,NaN,not_generated
2,M003,U003,2026-06-10,歩行状態に大きな変化はなく、現行の歩行支援を継続する。,大きな変化なし,現行支援を継続する,サ責B,True,2026-06-10,訪問,本人,"本人,日次記録","疲れやすいです,足が少しだるい",車椅子移動が複数回記録され、歩行訓練の実施有無が記録から確認できない。,計画目標に対する評価根拠が不足している可能性がある。,歩行状態、歩行訓練の実施状況、計画変更要否を確認する,NaN,pending_generation
3,M004,U004,2026-06-10,食事摂取量と体調に大きな変化はない。,大きな変化なし,現行支援を継続する,サ責A,True,2026-06-10,訪問,本人,"本人,日次記録,看護職申し送り","食欲が出ない,少しだるい",直近4回の記録で食事量低下。6月8日の昼食は3割程度。水分摂取量低下。発熱なし。,目標である食事量維持と体調安定について、達成できているとは判断しにくい。,看護職へ共有し、食事量、水分量、倦怠感を継続確認。計画変更要否を検討する,NaN,pending_generation
4,M005,U005,2026-06-10,生活援助を継続している。,大きな変化なし,現行支援を継続する,NaN,False,2026-06-10,訪問,本人,"本人,日次記録","部屋がきれいになって助かった,牛乳とパンを買いたい,体調は変わらない",床清掃、ごみ回収、買い物同行、体調確認を実施。体温36.4℃。発熱なし。,生活援助は概ね計画どおりだが、日次記録が簡潔すぎるため具体化が必要。,日次記録を具体化し、モニタリング承認者を入力する,NaN,pending_generation


,document_id,user_id,document_type,required,file_exists,valid_from,valid_to,approved,approved_by
0,D001,U001,訪問介護計画書,True,True,2026-06-01,2026-11-30,True,管理者A
1,D002,U001,モニタリング記録,True,True,2026-06-01,2026-06-30,True,サ責A
2,D003,U002,訪問介護計画書,True,True,2026-04-01,2026-09-30,True,管理者A
3,D004,U002,モニタリング記録,True,True,2026-05-01,2026-05-31,True,サ責A
4,D005,U003,訪問介護計画書,True,True,2026-03-01,2026-08-31,True,管理者B


In [6]:
display(
    document_status[
        [
            "document_id",
            "user_id",
            "document_type",
            "valid_from",
            "valid_to",
            "file_exists",
            "approved",
            "approved_by"
        ]
    ]
)

,document_id,user_id,document_type,valid_from,valid_to,file_exists,approved,approved_by
0,D001,U001,訪問介護計画書,2026-06-01,2026-11-30,True,True,管理者A
1,D002,U001,モニタリング記録,2026-06-01,2026-06-30,True,True,サ責A
2,D003,U002,訪問介護計画書,2026-04-01,2026-09-30,True,True,管理者A
3,D004,U002,モニタリング記録,2026-05-01,2026-05-31,True,True,サ責A
4,D005,U003,訪問介護計画書,2026-03-01,2026-08-31,True,True,管理者B
5,D006,U003,モニタリング記録,2026-06-01,2026-06-30,True,True,サ責B
6,D007,U004,訪問介護計画書,2026-05-01,2026-10-31,True,True,管理者A
7,D008,U004,モニタリング記録,2026-06-01,2026-06-30,True,True,サ責A
8,D009,U005,訪問介護計画書,2026-01-01,2026-06-30,True,True,管理者B
9,D010,U005,モニタリング記録,2026-06-01,2026-06-30,True,False,NaN


### 6. ルールベース判定


In [7]:

documents = document_status.copy()

In [8]:
# --------------------------------------------------
# 期限切れ・不足書類
# --------------------------------------------------

expired_documents = documents[
    (documents["required"] == True)
    & (
        (documents["file_exists"] == False)
        | (documents["valid_to"].isna())
        | (documents["valid_to"] < CHECK_DATE)
    )
]

print("=== 期限切れ・不足書類 ===")
display(expired_documents)


=== 期限切れ・不足書類 ===


,document_id,user_id,document_type,required,file_exists,valid_from,valid_to,approved,approved_by
3,D004,U002,モニタリング記録,True,True,2026-05-01,2026-05-31,True,サ責A


In [9]:
# --------------------------------------------------
# 未承認書類
# --------------------------------------------------

unapproved_documents = documents[
    (documents["required"] == True)
    & (documents["file_exists"] == True)
    & (documents["approved"] == False)
]

print("=== 未承認書類 ===")
display(unapproved_documents)

=== 未承認書類 ===


,document_id,user_id,document_type,required,file_exists,valid_from,valid_to,approved,approved_by
9,D010,U005,モニタリング記録,True,True,2026-06-01,2026-06-30,False,NaN


In [10]:
# --------------------------------------------------
# 特記事項が100文字未満の記録
# --------------------------------------------------

short_records = daily_records[
    daily_records["notes_length"] < 100
].copy()

print("=== 特記事項が100文字未満の記録 ===")
display(
    short_records[
        [
            "record_id",
            "user_id",
            "record_date",
            "special_notes",
            "notes_length",
            "user_quote",
            "observation",
            "numeric_data",
            "follow_up",
            "ai_draft_status",
        ]
    ]
)

=== 特記事項が100文字未満の記録 ===


,record_id,user_id,record_date,special_notes,notes_length,user_quote,observation,numeric_data,follow_up,ai_draft_status
0,R001,U001,2026-06-06,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,69,今日は大丈夫,立ち上がり時に軽度ふらつきあり。歩行器使用。転倒なし。,トイレ移動1回,次回も立ち上がり時のふらつきを確認する,not_generated
1,R002,U001,2026-06-08,トイレ移動時に歩行器を使用。声かけに応じてゆっくり移動され、転倒なく終了した。本人より体調不...,56,体調は悪くない,歩行器使用。声かけに応じて移動。転倒なし。,トイレ移動1回,現行支援を継続する,not_generated
2,R003,U001,2026-06-10,居室内の移動は歩行器で実施。立位保持は安定しており、排泄動作も普段通りであった。継続して安全...,52,いつも通りです,立位保持は安定。排泄動作は普段通り。,居室内移動1回,安全確認を継続する,not_generated
3,R004,U002,2026-06-06,朝食後の服薬を確認した。血圧は普段と大きな変化なく、倦怠感や頭痛の訴えはなかった。更衣時は一...,54,頭痛はないです,倦怠感なし。頭痛なし。更衣時に一部介助。,血圧128/74mmHg,服薬確認と体調観察を継続する,not_generated
4,R005,U002,2026-06-09,服薬忘れがないことを確認した。上衣の着脱時に介助を実施。本人より体調は変わらないとの発言があった。,49,体調は変わらない,服薬忘れなし。上衣着脱時に介助。,NaN,計画書の更新状況を確認する,not_generated
5,R006,U003,2026-06-05,居室から食堂まで車椅子で移動した。移乗時は職員が見守りを行い、安全に着座された。歩行訓練の実...,59,今日は少し疲れた,車椅子移動。移乗時は見守り。歩行訓練の実施有無は不明。,車椅子移動1回,歩行訓練の実施状況を確認する,not_generated
6,R007,U003,2026-06-07,食堂への移動は車椅子を使用した。本人は疲れやすいと話されていた。歩行の状態変化について追加確...,51,疲れやすいです,車椅子移動。疲労感の訴えあり。,車椅子移動1回,歩行状態と支援内容の変更要否を確認する,not_generated
7,R008,U003,2026-06-09,居室から浴室前まで車椅子で移動した。下肢の疲労感があるとの発言あり。歩行訓練を実施したかは記...,56,足が少しだるい,車椅子移動。下肢疲労感あり。歩行訓練の記録なし。,車椅子移動1回,歩行訓練の実施状況と計画見直し要否を確認する,not_generated
8,R009,U004,2026-06-04,昼食は主食5割、副食4割の摂取であった。普段より食欲が低下している様子。水分は声かけによりコ...,56,あまり食べたくない,食欲低下あり。水分は声かけで摂取。,"主食5割,副食4割,水分コップ1杯",食事量と水分量を継続観察する,not_generated
9,R010,U004,2026-06-06,朝食は半量程度で終了。本人より食欲が出ないとの訴えあり。倦怠感がみられたため、看護職へ申し送...,52,食欲が出ない,倦怠感あり。看護職へ申し送り。,朝食5割,看護職と共有し継続観察する,not_generated


In [11]:
# --------------------------------------------------
# モニタリング必須項目の不足チェック
# --------------------------------------------------

required_monitoring_fields = [
    "interview_date",
    "interview_method",
    "attendees",
    "information_sources",
    "user_quotes",
    "objective_facts",
    "goal_evaluation",
    "future_action",
]

missing_monitoring_fields = monitoring_records.copy()

for col in required_monitoring_fields:
    missing_monitoring_fields[f"{col}_missing"] = (
        missing_monitoring_fields[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

missing_columns = [
    f"{col}_missing"
    for col in required_monitoring_fields
]

missing_monitoring_fields["missing_count"] = (
    missing_monitoring_fields[missing_columns]
    .sum(axis=1)
)

print("=== モニタリング必須項目の不足 ===")
display(
    missing_monitoring_fields[
        [
            "monitoring_id",
            "user_id",
            "missing_count",
            "ai_draft_status",
        ]
    ]
)

=== モニタリング必須項目の不足 ===


,monitoring_id,user_id,missing_count,ai_draft_status
0,M001,U001,0,not_generated
1,M002,U002,0,not_generated
2,M003,U003,0,pending_generation
3,M004,U004,0,pending_generation
4,M005,U005,0,pending_generation


### 7. 利用者別アラート一覧


#### 利用者別アラート表示の考え方

本MVPでは、利用者別の確認優先度を0〜100点などのスコアでは表示しない。

未完・未承認・未確認の書類は、最終的に期限までにすべて対応が必要であるため、点数化よりも「どの利用者の、どの書類・記録に対応が必要か」を明確に表示する。

そのため、利用者ごとに以下の状態別で対象書類名を表示する。

- 未作成
- 未承認
- 未確認・確認不足

さらに、同じ書類種別で複数件のアラートがある場合は、`日々の介護記録（4件）` のように件数を併記する。
Streamlit化を見据え、クリック先の材料となる `target_type`、`target_id`、`action_type` を `alerts_integrated.csv` に保持し、利用者別サマリーにも対象ID一覧をJSON形式で保存する。

これにより、職員が次に確認・修正・保存すべき対象を直感的に把握できるようにする。


In [12]:
# ==================================================
# 7. 利用者別アラート一覧
# ==================================================

alerts = []

# --------------------------------------------------
# 1. document_status から書類系アラートを作成
# --------------------------------------------------

for _, row in document_status.iterrows():
    document_id = row["document_id"]
    user_id = row["user_id"]
    document_type = row["document_type"]

    required = to_bool(row["required"])
    file_exists = to_bool(row["file_exists"])
    approved = to_bool(row["approved"])

    valid_from = pd.to_datetime(row["valid_from"], errors="coerce")
    valid_to = pd.to_datetime(row["valid_to"], errors="coerce")

    # 必須書類なのにファイルが存在しない場合
    if required and not file_exists:
        if document_type == "モニタリング記録":
            action_type = "generate_monitoring_draft"
            severity = "high"
        elif document_type == "訪問介護計画書":
            action_type = "review_care_plan"
            severity = "high"
        else:
            action_type = "create_document_draft"
            severity = "medium"

        alerts.append({
            "user_id": user_id,
            "alert_category": "document",
            "target_type": "document",
            "target_id": document_id,
            "document_type": document_type,
            "alert_type": "書類未作成",
            "severity": severity,
            "detail": "必須書類のファイルが存在しません。",
            "action_type": action_type,
            "human_review_required": True,
        })

    # 訪問介護計画書の有効期限を確認
    if document_type == "訪問介護計画書":
        is_invalid_period = (
            pd.isna(valid_from)
            or pd.isna(valid_to)
            or not (valid_from <= CHECK_DATE <= valid_to)
        )

        if is_invalid_period:
            alerts.append({
                "user_id": user_id,
                "alert_category": "document",
                "target_type": "care_plan",
                "target_id": document_id,
                "document_type": document_type,
                "alert_type": "計画書の有効期限外",
                "severity": "high",
                "detail": f"{CHECK_DATE.date()} 時点で有効な計画書ではありません。",
                "action_type": "review_care_plan",
                "human_review_required": True,
            })

    # モニタリング記録が当月分として有効か確認
    if document_type == "モニタリング記録":
        is_invalid_period = (
            pd.isna(valid_from)
            or pd.isna(valid_to)
            or not (valid_from <= CHECK_DATE <= valid_to)
        )

        if is_invalid_period:
            alerts.append({
                "user_id": user_id,
                "alert_category": "document",
                "target_type": "monitoring",
                "target_id": document_id,
                "document_type": document_type,
                "alert_type": "当月モニタリング未確認",
                "severity": "high",
                "detail": f"{CHECK_DATE.year}年{CHECK_DATE.month}月分のモニタリング記録が確認できません。",
                "action_type": "generate_monitoring_draft",
                "human_review_required": True,
            })

    # ファイルはあるが、承認されていない場合
    if file_exists and not approved:
        alerts.append({
            "user_id": user_id,
            "alert_category": "document",
            "target_type": "document",
            "target_id": document_id,
            "document_type": document_type,
            "alert_type": "未承認",
            "severity": "medium",
            "detail": "書類は存在しますが、担当者による承認が完了していません。",
            "action_type": "review_document",
            "human_review_required": True,
        })


# --------------------------------------------------
# 2. daily_records から特記事項不足アラートを作成
# --------------------------------------------------

for _, row in short_records.iterrows():
    alerts.append({
        "user_id": row["user_id"],
        "alert_category": "daily_record",
        "target_type": "daily_record",
        "target_id": row["record_id"],
        "document_type": "日々の介護記録",
        "alert_type": "特記事項不足",
        "severity": "medium",
        "detail": (
            f"特記事項が100文字未満です。"
            f"現在の文字数: {int(row['notes_length'])}文字"
        ),
        "action_type": "generate_note_draft",
        "human_review_required": True,
    })


# --------------------------------------------------
# 3. monitoring_records から必須項目不足アラートを作成
# --------------------------------------------------

for _, row in missing_monitoring_fields.iterrows():
    missing_count = int(row["missing_count"])

    if missing_count == 0:
        continue

    missing_field_names = []

    for col in required_monitoring_fields:
        missing_col = f"{col}_missing"

        if row[missing_col]:
            missing_field_names.append(col)

    alerts.append({
        "user_id": row["user_id"],
        "alert_category": "monitoring",
        "target_type": "monitoring",
        "target_id": row["monitoring_id"],
        "document_type": "モニタリング記録",
        "alert_type": "モニタリング必須項目不足",
        "severity": "medium",
        "detail": (
            f"モニタリング記録の必須項目が {missing_count} 件不足しています。"
            f"不足項目: {', '.join(missing_field_names)}"
        ),
        "action_type": "generate_monitoring_draft",
        "human_review_required": True,
    })


# --------------------------------------------------
# 4. alerts_df を作成
# --------------------------------------------------

alerts_df = pd.DataFrame(alerts)

# 表示順を整える
display_columns = [
    "user_id",
    "alert_category",
    "alert_type",
    "severity",
    "document_type",
    "target_type",
    "target_id",
    "detail",
    "action_type",
    "human_review_required",
]

if alerts_df.empty:
    print("✅ コンプライアンス上の警告はありません。")
    alerts_df = pd.DataFrame(columns=display_columns)
else:
    alerts_df = alerts_df[display_columns].sort_values(
        by=["user_id", "severity", "alert_category", "alert_type"],
        ascending=[True, True, True, True]
    )

    print(f"⚠️ 要確認のアラートが {len(alerts_df)} 件あります。")
    display(alerts_df)



# --------------------------------------------------
# 5. 利用者別に、状態別の対象書類名・対象IDを集計
# --------------------------------------------------
# 点数方式の優先度表示は行わない。
# 現場で次に必要なのは「何点か」ではなく、
# 「どの利用者の、どの書類・記録を確認すべきか」であるため。

UNCREATED_ALERT_TYPES = [
    "書類未作成",
]

UNAPPROVED_ALERT_TYPES = [
    "未承認",
]

UNCONFIRMED_ALERT_TYPES = [
    "当月モニタリング未確認",
    "計画書の有効期限外",
    "モニタリング必須項目不足",
    "特記事項不足",
]


def format_document_counts(user_alerts: pd.DataFrame, alert_types: list[str]) -> str:
    """
    指定したalert_typeに該当する書類・記録名を件数付きで表示する。
    例：日々の介護記録（4件）、モニタリング記録（1件）
    対象がない場合は「なし」と表示する。
    """
    target_alerts = user_alerts[
        user_alerts["alert_type"].isin(alert_types)
    ].copy()

    if target_alerts.empty:
        return "なし"

    document_counts = (
        target_alerts
        .groupby("document_type")
        .size()
        .reset_index(name="count")
        .sort_values(
            by=["count", "document_type"],
            ascending=[False, True]
        )
    )

    return "、".join(
        [
            f"{row['document_type']}（{int(row['count'])}件）"
            for _, row in document_counts.iterrows()
        ]
    )


def targets_json_for_alert_types(user_alerts: pd.DataFrame, alert_types: list[str]) -> str:
    """
    Streamlit化を見据え、クリック先として使える対象情報をJSON文字列で保存する。
    alerts_integrated.csvにも同じ粒度の行が残るが、利用者別サマリーからも参照できるようにする。
    """
    target_alerts = user_alerts[
        user_alerts["alert_type"].isin(alert_types)
    ].copy()

    if target_alerts.empty:
        return "[]"

    target_columns = [
        "alert_type",
        "document_type",
        "target_type",
        "target_id",
        "action_type",
        "detail",
    ]

    targets = (
        target_alerts[target_columns]
        .fillna("")
        .astype(str)
        .to_dict(orient="records")
    )

    return json.dumps(
        targets,
        ensure_ascii=False,
    )


summary_rows = []

if not alerts_df.empty:
    for user_id, user_alerts in alerts_df.groupby("user_id"):
        user_name_rows = users.loc[
            users["user_id"] == user_id,
            "user_name"
        ]

        user_name = (
            user_name_rows.iloc[0]
            if len(user_name_rows) > 0
            else ""
        )

        uncreated_count = int(
            user_alerts["alert_type"].isin(UNCREATED_ALERT_TYPES).sum()
        )

        unapproved_count = int(
            user_alerts["alert_type"].isin(UNAPPROVED_ALERT_TYPES).sum()
        )

        unconfirmed_count = int(
            user_alerts["alert_type"].isin(UNCONFIRMED_ALERT_TYPES).sum()
        )

        alert_count = int(len(user_alerts))

        uncreated_documents = format_document_counts(
            user_alerts,
            UNCREATED_ALERT_TYPES,
        )

        unapproved_documents = format_document_counts(
            user_alerts,
            UNAPPROVED_ALERT_TYPES,
        )

        unconfirmed_documents = format_document_counts(
            user_alerts,
            UNCONFIRMED_ALERT_TYPES,
        )

        uncreated_targets_json = targets_json_for_alert_types(
            user_alerts,
            UNCREATED_ALERT_TYPES,
        )

        unapproved_targets_json = targets_json_for_alert_types(
            user_alerts,
            UNAPPROVED_ALERT_TYPES,
        )

        unconfirmed_targets_json = targets_json_for_alert_types(
            user_alerts,
            UNCONFIRMED_ALERT_TYPES,
        )

        alert_message = (
            f"{user_name}（{user_id}）には、対応が必要な書類・記録が{alert_count}件あります。"
            f"未作成：{uncreated_count}件（{uncreated_documents}）。"
            f"未承認：{unapproved_count}件（{unapproved_documents}）。"
            f"未確認・確認不足：{unconfirmed_count}件（{unconfirmed_documents}）。"
            "職員による確認・修正・保存を行ってください。"
        )

        summary_rows.append({
            "user_id": user_id,
            "user_name": user_name,
            "alert_count": alert_count,
            "uncreated_count": uncreated_count,
            "unapproved_count": unapproved_count,
            "unconfirmed_count": unconfirmed_count,
            "uncreated_documents": uncreated_documents,
            "unapproved_documents": unapproved_documents,
            "unconfirmed_documents": unconfirmed_documents,
            "uncreated_targets_json": uncreated_targets_json,
            "unapproved_targets_json": unapproved_targets_json,
            "unconfirmed_targets_json": unconfirmed_targets_json,
            "alert_message": alert_message,
        })

alert_summary_by_user = pd.DataFrame(summary_rows)

if alert_summary_by_user.empty:
    print("✅ 利用者別に集計すべきアラートはありません。")
else:
    alert_summary_by_user = alert_summary_by_user.sort_values(
        by=[
            "uncreated_count",
            "unapproved_count",
            "unconfirmed_count",
            "alert_count",
            "user_id",
        ],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)

    print("=== 利用者別アラート一覧（状態別の対象書類名・件数） ===")
    display(alert_summary_by_user)

    print("=== アラート文章サンプル ===")

    for message in alert_summary_by_user["alert_message"].head(5):
        print("-", message)


# --------------------------------------------------
# 6. 統合アラート一覧をCSVとして保存
# --------------------------------------------------

alerts_output_path = BASE_DIR / "alerts_integrated.csv"

alerts_df.to_csv(
    alerts_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ 統合アラート一覧をCSVとして保存しました。")
print("保存先:", alerts_output_path)


# --------------------------------------------------
# 7. 利用者別アラート一覧をCSVとして保存
# --------------------------------------------------

alert_summary_output_path = BASE_DIR / "alert_summary_by_user.csv"

alert_summary_by_user.to_csv(
    alert_summary_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ 利用者別アラート一覧をCSVとして保存しました。")
print("保存先:", alert_summary_output_path)


⚠️ 要確認のアラートが 17 件あります。


,user_id,alert_category,alert_type,severity,document_type,target_type,target_id,detail,action_type,human_review_required
2,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R001,特記事項が100文字未満です。現在の文字数: 69文字,generate_note_draft,True
3,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R002,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
4,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R003,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft,True
0,U002,document,当月モニタリング未確認,high,モニタリング記録,monitoring,D004,2026年6月分のモニタリング記録が確認できません。,generate_monitoring_draft,True
5,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R004,特記事項が100文字未満です。現在の文字数: 54文字,generate_note_draft,True
6,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R005,特記事項が100文字未満です。現在の文字数: 49文字,generate_note_draft,True
7,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R006,特記事項が100文字未満です。現在の文字数: 59文字,generate_note_draft,True
8,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R007,特記事項が100文字未満です。現在の文字数: 51文字,generate_note_draft,True
9,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R008,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
10,U004,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R009,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True


=== 利用者別アラート一覧（状態別の対象書類名・件数） ===


,user_id,user_name,alert_count,uncreated_count,unapproved_count,unconfirmed_count,uncreated_documents,unapproved_documents,unconfirmed_documents,uncreated_targets_json,unapproved_targets_json,unconfirmed_targets_json,alert_message
0,U005,利用者E,4,0,1,3,なし,モニタリング記録（1件）,日々の介護記録（3件）,[],"[{""alert_type"": ""未承認"", ""document_type"": ""モニタリン...","[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者E（U005）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
1,U004,利用者D,4,0,0,4,なし,なし,日々の介護記録（4件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者D（U004）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
2,U001,利用者A,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者A（U001）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
3,U002,利用者B,3,0,0,3,なし,なし,日々の介護記録（2件）、モニタリング記録（1件）,[],[],"[{""alert_type"": ""当月モニタリング未確認"", ""document_type""...",利用者B（U002）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
4,U003,利用者C,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者C（U003）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...


=== アラート文章サンプル ===
- 利用者E（U005）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認：1件（モニタリング記録（1件））。未確認・確認不足：3件（日々の介護記録（3件））。職員による確認・修正・保存を行ってください。
- 利用者D（U004）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認：0件（なし）。未確認・確認不足：4件（日々の介護記録（4件））。職員による確認・修正・保存を行ってください。
- 利用者A（U001）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認：0件（なし）。未確認・確認不足：3件（日々の介護記録（3件））。職員による確認・修正・保存を行ってください。
- 利用者B（U002）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認：0件（なし）。未確認・確認不足：3件（日々の介護記録（2件）、モニタリング記録（1件））。職員による確認・修正・保存を行ってください。
- 利用者C（U003）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認：0件（なし）。未確認・確認不足：3件（日々の介護記録（3件））。職員による確認・修正・保存を行ってください。
✅ 統合アラート一覧をCSVとして保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/alerts_integrated.csv
✅ 利用者別アラート一覧をCSVとして保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/alert_summary_by_user.csv


#### alerts_integrated.csv と alert_summary_by_user.csv の保存確認

In [13]:
alerts_output_path = BASE_DIR / "alerts_integrated.csv"
alert_summary_output_path = BASE_DIR / "alert_summary_by_user.csv"

# --------------------------------------------------
# 1. alerts_integrated.csv の保存確認
# --------------------------------------------------

if alerts_output_path.exists():
    print("✅ alerts_integrated.csv は作成されています。")
    print("保存先:", alerts_output_path)

    saved_alerts = pd.read_csv(alerts_output_path)

    print("alerts_integrated.csv の行数:", len(saved_alerts))
    print("alerts_integrated.csv の列数:", len(saved_alerts.columns))

    display(saved_alerts.head())
else:
    print("⚠️ alerts_integrated.csv が見つかりません。")
    print("確認したパス:", alerts_output_path)


# --------------------------------------------------
# 2. alert_summary_by_user.csv の保存確認
# --------------------------------------------------

if alert_summary_output_path.exists():
    print("✅ alert_summary_by_user.csv は作成されています。")
    print("保存先:", alert_summary_output_path)

    saved_alert_summary = pd.read_csv(alert_summary_output_path)

    print("alert_summary_by_user.csv の行数:", len(saved_alert_summary))
    print("alert_summary_by_user.csv の列数:", len(saved_alert_summary.columns))

    display(saved_alert_summary.head())
else:
    print("⚠️ alert_summary_by_user.csv が見つかりません。")
    print("確認したパス:", alert_summary_output_path)


✅ alerts_integrated.csv は作成されています。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/alerts_integrated.csv
alerts_integrated.csv の行数: 17
alerts_integrated.csv の列数: 10


,user_id,alert_category,alert_type,severity,document_type,target_type,target_id,detail,action_type,human_review_required
0,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R001,特記事項が100文字未満です。現在の文字数: 69文字,generate_note_draft,True
1,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R002,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
2,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R003,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft,True
3,U002,document,当月モニタリング未確認,high,モニタリング記録,monitoring,D004,2026年6月分のモニタリング記録が確認できません。,generate_monitoring_draft,True
4,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R004,特記事項が100文字未満です。現在の文字数: 54文字,generate_note_draft,True


✅ alert_summary_by_user.csv は作成されています。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/alert_summary_by_user.csv
alert_summary_by_user.csv の行数: 5
alert_summary_by_user.csv の列数: 13


,user_id,user_name,alert_count,uncreated_count,unapproved_count,unconfirmed_count,uncreated_documents,unapproved_documents,unconfirmed_documents,uncreated_targets_json,unapproved_targets_json,unconfirmed_targets_json,alert_message
0,U005,利用者E,4,0,1,3,なし,モニタリング記録（1件）,日々の介護記録（3件）,[],"[{""alert_type"": ""未承認"", ""document_type"": ""モニタリン...","[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者E（U005）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
1,U004,利用者D,4,0,0,4,なし,なし,日々の介護記録（4件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者D（U004）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
2,U001,利用者A,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者A（U001）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
3,U002,利用者B,3,0,0,3,なし,なし,日々の介護記録（2件）、モニタリング記録（1件）,[],[],"[{""alert_type"": ""当月モニタリング未確認"", ""document_type""...",利用者B（U002）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
4,U003,利用者C,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者C（U003）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...


### 8. 特記事項補填案Payload作成

In [14]:
# ==================================================
# 8. 特記事項補填案Payload作成
# ==================================================

# --------------------------------------------------
# 1. 特記事項補填案の対象アラートを抽出
# --------------------------------------------------

note_draft_alerts = alerts_df[
    alerts_df["action_type"] == "generate_note_draft"
].copy()

print("=== 特記事項補填案の対象アラート ===")
print("対象件数:", len(note_draft_alerts))

display(note_draft_alerts)

=== 特記事項補填案の対象アラート ===
対象件数: 15


,user_id,alert_category,alert_type,severity,document_type,target_type,target_id,detail,action_type,human_review_required
2,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R001,特記事項が100文字未満です。現在の文字数: 69文字,generate_note_draft,True
3,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R002,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
4,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R003,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft,True
5,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R004,特記事項が100文字未満です。現在の文字数: 54文字,generate_note_draft,True
6,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R005,特記事項が100文字未満です。現在の文字数: 49文字,generate_note_draft,True
7,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R006,特記事項が100文字未満です。現在の文字数: 59文字,generate_note_draft,True
8,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R007,特記事項が100文字未満です。現在の文字数: 51文字,generate_note_draft,True
9,U003,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R008,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
10,U004,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R009,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
11,U004,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R010,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft,True


In [15]:
# --------------------------------------------------
# 2. 対象アラートに日次記録の詳細を結合
# --------------------------------------------------

note_draft_targets = note_draft_alerts.merge(
    daily_records,
    left_on="target_id",
    right_on="record_id",
    how="left",
    suffixes=("_alert", "_record")
)

print("=== 特記事項補填案の対象記録 ===")
print("対象件数:", len(note_draft_targets))

display(
    note_draft_targets[
        [
            "user_id_alert",
            "record_id",
            "record_date",
            "service_items",
            "special_notes",
            "notes_length",
            "user_quote",
            "observation",
            "numeric_data",
            "follow_up",
            "detail",
            "action_type",
        ]
    ]
)

=== 特記事項補填案の対象記録 ===
対象件数: 15


,user_id_alert,record_id,record_date,service_items,special_notes,notes_length,user_quote,observation,numeric_data,follow_up,detail,action_type
0,U001,R001,2026-06-06,"歩行見守り,排泄介助",居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,69,今日は大丈夫,立ち上がり時に軽度ふらつきあり。歩行器使用。転倒なし。,トイレ移動1回,次回も立ち上がり時のふらつきを確認する,特記事項が100文字未満です。現在の文字数: 69文字,generate_note_draft
1,U001,R002,2026-06-08,"歩行見守り,排泄介助",トイレ移動時に歩行器を使用。声かけに応じてゆっくり移動され、転倒なく終了した。本人より体調不...,56,体調は悪くない,歩行器使用。声かけに応じて移動。転倒なし。,トイレ移動1回,現行支援を継続する,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft
2,U001,R003,2026-06-10,"歩行見守り,排泄介助",居室内の移動は歩行器で実施。立位保持は安定しており、排泄動作も普段通りであった。継続して安全...,52,いつも通りです,立位保持は安定。排泄動作は普段通り。,居室内移動1回,安全確認を継続する,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft
3,U002,R004,2026-06-06,"服薬確認,体調観察",朝食後の服薬を確認した。血圧は普段と大きな変化なく、倦怠感や頭痛の訴えはなかった。更衣時は一...,54,頭痛はないです,倦怠感なし。頭痛なし。更衣時に一部介助。,血圧128/74mmHg,服薬確認と体調観察を継続する,特記事項が100文字未満です。現在の文字数: 54文字,generate_note_draft
4,U002,R005,2026-06-09,"服薬確認,更衣介助",服薬忘れがないことを確認した。上衣の着脱時に介助を実施。本人より体調は変わらないとの発言があった。,49,体調は変わらない,服薬忘れなし。上衣着脱時に介助。,NaN,計画書の更新状況を確認する,特記事項が100文字未満です。現在の文字数: 49文字,generate_note_draft
5,U003,R006,2026-06-05,移動介助,居室から食堂まで車椅子で移動した。移乗時は職員が見守りを行い、安全に着座された。歩行訓練の実...,59,今日は少し疲れた,車椅子移動。移乗時は見守り。歩行訓練の実施有無は不明。,車椅子移動1回,歩行訓練の実施状況を確認する,特記事項が100文字未満です。現在の文字数: 59文字,generate_note_draft
6,U003,R007,2026-06-07,移動介助,食堂への移動は車椅子を使用した。本人は疲れやすいと話されていた。歩行の状態変化について追加確...,51,疲れやすいです,車椅子移動。疲労感の訴えあり。,車椅子移動1回,歩行状態と支援内容の変更要否を確認する,特記事項が100文字未満です。現在の文字数: 51文字,generate_note_draft
7,U003,R008,2026-06-09,移動介助,居室から浴室前まで車椅子で移動した。下肢の疲労感があるとの発言あり。歩行訓練を実施したかは記...,56,足が少しだるい,車椅子移動。下肢疲労感あり。歩行訓練の記録なし。,車椅子移動1回,歩行訓練の実施状況と計画見直し要否を確認する,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft
8,U004,R009,2026-06-04,"食事見守り,水分摂取",昼食は主食5割、副食4割の摂取であった。普段より食欲が低下している様子。水分は声かけによりコ...,56,あまり食べたくない,食欲低下あり。水分は声かけで摂取。,"主食5割,副食4割,水分コップ1杯",食事量と水分量を継続観察する,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft
9,U004,R010,2026-06-06,"食事見守り,体調観察",朝食は半量程度で終了。本人より食欲が出ないとの訴えあり。倦怠感がみられたため、看護職へ申し送...,52,食欲が出ない,倦怠感あり。看護職へ申し送り。,朝食5割,看護職と共有し継続観察する,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft


In [16]:
# --------------------------------------------------
# 3. 特記事項補填案生成用payloadを作成する関数
# --------------------------------------------------

def build_note_draft_payload(
    record_id: str,
) -> dict:
    """
    特記事項が短い日次記録について、
    AI補填案生成に使うpayloadを作成する。

    注意：
    AIは新しい事実を作らず、既存の記録項目だけを使う。
    """

    target_records = daily_records[
        daily_records["record_id"] == record_id
    ].copy()

    if target_records.empty:
        raise ValueError(
            f"record_id {record_id} は daily_records.csv に存在しません。"
        )

    if len(target_records) > 1:
        raise ValueError(
            f"record_id {record_id} が複数存在します。"
        )

    row = target_records.iloc[0]

    payload = {
        "task": "generate_note_draft",
        "record_id": clean_text(row["record_id"]),
        "user_id": clean_text(row["user_id"]),
        "record_date": row["record_date"].strftime("%Y-%m-%d"),
        "service_items": split_items(row["service_items"]),
        "current_special_notes": clean_text(row["special_notes"]),
        "notes_length": int(row["notes_length"]),
        "source_facts": {
            "user_quote": clean_text(row["user_quote"]),
            "observation": clean_text(row["observation"]),
            "numeric_data": clean_text(row["numeric_data"]),
            "follow_up": clean_text(row["follow_up"]),
        },
        "generation_rules": [
            "記録に存在する事実だけを使用する。",
            "本人発言は user_quote に記載された内容だけを使用する。",
            "観察事項は observation に記載された内容だけを使用する。",
            "数値情報は numeric_data に記載された内容だけを使用する。",
            "今後の対応は follow_up に記載された内容だけを使用する。",
            "存在しない支援、発言、症状、判断を推測して追加しない。",
            "医学的診断を行わない。",
            "職員が確認・修正する前提の下書きとして作成する。",
        ],
        "required_output": {
            "language": "Japanese",
            "draft_length": "100〜200文字程度",
            "human_review_required": True,
            "output_sections": [
                "補填案",
                "使用した根拠",
                "職員確認が必要な事項",
            ],
        },
    }

    return payload

In [17]:
# --------------------------------------------------
# 4. 先頭1件でpayloadを確認
# --------------------------------------------------

target_record_id = note_draft_alerts.iloc[0]["target_id"]

note_payload = build_note_draft_payload(
    record_id=target_record_id
)

print(
    json.dumps(
        note_payload,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "task": "generate_note_draft",
  "record_id": "R001",
  "user_id": "U001",
  "record_date": "2026-06-06",
  "service_items": [
    "歩行見守り",
    "排泄介助"
  ],
  "current_special_notes": "居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見守りを実施した。排泄後は安全に居室へ戻られた。",
  "notes_length": 69,
  "source_facts": {
    "user_quote": "今日は大丈夫",
    "observation": "立ち上がり時に軽度ふらつきあり。歩行器使用。転倒なし。",
    "numeric_data": "トイレ移動1回",
    "follow_up": "次回も立ち上がり時のふらつきを確認する"
  },
  "generation_rules": [
    "記録に存在する事実だけを使用する。",
    "本人発言は user_quote に記載された内容だけを使用する。",
    "観察事項は observation に記載された内容だけを使用する。",
    "数値情報は numeric_data に記載された内容だけを使用する。",
    "今後の対応は follow_up に記載された内容だけを使用する。",
    "存在しない支援、発言、症状、判断を推測して追加しない。",
    "医学的診断を行わない。",
    "職員が確認・修正する前提の下書きとして作成する。"
  ],
  "required_output": {
    "language": "Japanese",
    "draft_length": "100〜200文字程度",
    "human_review_required": true,
    "output_sections": [
      "補填案",
      "使用した根拠",
      "職員確認が必要な事項"
    ]
  }
}


In [18]:
# --------------------------------------------------
# 5. 特記事項補填案生成用プロンプトを作成
# --------------------------------------------------

source_facts = note_payload["source_facts"]

note_draft_prompt = f"""
あなたは訪問介護事業所の記録作成を支援するAIです。
以下の日次記録をもとに、特記事項の補填案を作成してください。

【重要事項】
- 記録に存在しない事実を推測して追加しないでください。
- 医学的な診断を行わないでください。
- 本人発言は、記録にある発言のみ使用してください。
- 現在の特記事項を大きく書き換えるのではなく、不足している根拠情報を補う形で補填案を作成してください。
- 下書きは必ず職員が確認・修正する前提で作成してください。
- 情報が不足している場合は、職員確認が必要な事項として明記してください。

【記録ID】
{note_payload["record_id"]}

【利用者ID】
{note_payload["user_id"]}

【記録日】
{note_payload["record_date"]}

【サービス内容】
{note_payload["service_items"]}

【現在の特記事項】
{note_payload["current_special_notes"]}

【利用者の発言】
{source_facts["user_quote"]}

【観察事項】
{source_facts["observation"]}

【数値情報】
{source_facts["numeric_data"]}

【今後の対応】
{source_facts["follow_up"]}

【出力項目】
1. 特記事項の補填案
2. 使用した根拠
3. 職員確認が必要な事項
""".strip()

print(note_draft_prompt)

あなたは訪問介護事業所の記録作成を支援するAIです。
以下の日次記録をもとに、特記事項の補填案を作成してください。

【重要事項】
- 記録に存在しない事実を推測して追加しないでください。
- 医学的な診断を行わないでください。
- 本人発言は、記録にある発言のみ使用してください。
- 現在の特記事項を大きく書き換えるのではなく、不足している根拠情報を補う形で補填案を作成してください。
- 下書きは必ず職員が確認・修正する前提で作成してください。
- 情報が不足している場合は、職員確認が必要な事項として明記してください。

【記録ID】
R001

【利用者ID】
U001

【記録日】
2026-06-06

【サービス内容】
['歩行見守り', '排泄介助']

【現在の特記事項】
居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見守りを実施した。排泄後は安全に居室へ戻られた。

【利用者の発言】
今日は大丈夫

【観察事項】
立ち上がり時に軽度ふらつきあり。歩行器使用。転倒なし。

【数値情報】
トイレ移動1回

【今後の対応】
次回も立ち上がり時のふらつきを確認する

【出力項目】
1. 特記事項の補填案
2. 使用した根拠
3. 職員確認が必要な事項


### 9. 特記事項補填案サンプル作成

In [19]:
# ==================================================
# 9. 特記事項補填案サンプル作成
# ==================================================

note_generated_draft = """
【特記事項の補填案】
居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふらつきが見られたため、側方より見守りを実施した。本人より「今日は大丈夫」との発言があり、排泄後は安全に居室へ戻られた。転倒は確認されていない。次回も立ち上がり時のふらつきの有無を確認する。

【使用した根拠】
- 現在の特記事項：歩行器を使用して移動、立ち上がり時に軽度のふらつき、側方より見守り、排泄後は安全に居室へ戻った
- 利用者の発言：「今日は大丈夫」
- 観察事項：歩行器使用、転倒なし
- 数値情報：トイレ移動1回
- 今後の対応：次回も立ち上がり時のふらつきを確認する

【職員確認が必要な事項】
- ふらつきの程度や発生場面の記載が適切か
- 見守り方法の表現が実際の支援内容と一致しているか
- 排泄介助後の状態に追加記録が必要か
""".strip()

print(note_generated_draft)

【特記事項の補填案】
居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふらつきが見られたため、側方より見守りを実施した。本人より「今日は大丈夫」との発言があり、排泄後は安全に居室へ戻られた。転倒は確認されていない。次回も立ち上がり時のふらつきの有無を確認する。

【使用した根拠】
- 現在の特記事項：歩行器を使用して移動、立ち上がり時に軽度のふらつき、側方より見守り、排泄後は安全に居室へ戻った
- 利用者の発言：「今日は大丈夫」
- 観察事項：歩行器使用、転倒なし
- 数値情報：トイレ移動1回
- 今後の対応：次回も立ち上がり時のふらつきを確認する

【職員確認が必要な事項】
- ふらつきの程度や発生場面の記載が適切か
- 見守り方法の表現が実際の支援内容と一致しているか
- 排泄介助後の状態に追加記録が必要か


### 10. 特記事項補填案保存

In [20]:
# ==================================================
# 10. 特記事項補填案保存
# ==================================================

# --------------------------------------------------
# 1. 対象record_idを指定
# --------------------------------------------------

target_record_id = note_payload["record_id"]

target_mask = daily_records["record_id"] == target_record_id

if target_mask.sum() == 0:
    raise ValueError(
        f"{target_record_id} の日次記録が見つかりません。"
    )

if target_mask.sum() > 1:
    raise ValueError(
        f"{target_record_id} の日次記録が複数あります。"
    )

# --------------------------------------------------
# 2. 正式な特記事項は上書きせず、AI補填案用の列に保存
# --------------------------------------------------

daily_records.loc[target_mask, "ai_note_draft"] = note_generated_draft
daily_records.loc[target_mask, "ai_note_draft_status"] = "generated_pending_review"
daily_records.loc[target_mask, "ai_note_draft_source"] = "generate_note_draft"
daily_records.loc[target_mask, "ai_note_draft_created_at"] = "2026-06-10 17:30"

print("✅ 特記事項補填案を一時保存しました。")
print("状態: generated_pending_review")

display(
    daily_records.loc[
        target_mask,
        [
            "record_id",
            "user_id",
            "record_date",
            "special_notes",
            "notes_length",
            "ai_note_draft_status",
            "ai_note_draft",
        ]
    ]
)

✅ 特記事項補填案を一時保存しました。
状態: generated_pending_review


,record_id,user_id,record_date,special_notes,notes_length,ai_note_draft_status,ai_note_draft
0,R001,U001,2026-06-06,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,69,generated_pending_review,【特記事項の補填案】\n居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふら...


In [21]:
# --------------------------------------------------
# 3. 特記事項補填案入りCSVを別名保存
# --------------------------------------------------

note_draft_output_path = BASE_DIR / "daily_records_with_ai_note_draft.csv"

daily_records.to_csv(
    note_draft_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ 特記事項補填案入りCSVを別名で保存しました。")
print("保存先:", note_draft_output_path)

✅ 特記事項補填案入りCSVを別名で保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/daily_records_with_ai_note_draft.csv


In [22]:
# --------------------------------------------------
# 4. 保存結果を確認
# --------------------------------------------------

if note_draft_output_path.exists():
    print("✅ daily_records_with_ai_note_draft.csv は作成されています。")
    print("保存先:", note_draft_output_path)

    saved_note_records = pd.read_csv(note_draft_output_path)

    print("行数:", len(saved_note_records))
    print("列数:", len(saved_note_records.columns))

    display(
        saved_note_records[
            saved_note_records["record_id"] == target_record_id
        ][
            [
                "record_id",
                "user_id",
                "record_date",
                "special_notes",
                "ai_note_draft_status",
                "ai_note_draft",
            ]
        ]
    )
else:
    print("⚠️ daily_records_with_ai_note_draft.csv が見つかりません。")
    print("確認したパス:", note_draft_output_path)

✅ daily_records_with_ai_note_draft.csv は作成されています。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/daily_records_with_ai_note_draft.csv
行数: 15
列数: 20


,record_id,user_id,record_date,special_notes,ai_note_draft_status,ai_note_draft
0,R001,U001,2026-06-06,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,generated_pending_review,【特記事項の補填案】\n居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふら...


### 11. 特記事項補填案の職員確認済み保存

In [23]:
# --------------------------------------------------
# 1. 特記事項補填案入りCSVを読み込む
# --------------------------------------------------

note_review_source_path = BASE_DIR / "daily_records_with_ai_note_draft.csv"

if not note_review_source_path.exists():
    raise FileNotFoundError(
        "daily_records_with_ai_note_draft.csv が見つかりません。"
    )

note_review_records = pd.read_csv(note_review_source_path)

print("✅ 特記事項補填案入りCSVを読み込みました。")
print("読み込み元:", note_review_source_path)

# --------------------------------------------------
# 2. 対象record_idを確認
# --------------------------------------------------

target_record_id = note_payload["record_id"]

target_mask = note_review_records["record_id"] == target_record_id

if target_mask.sum() == 0:
    raise ValueError(
        f"{target_record_id} の日次記録が見つかりません。"
    )

if target_mask.sum() > 1:
    raise ValueError(
        f"{target_record_id} の日次記録が複数あります。"
    )

# --------------------------------------------------
# 3. AI補填案を職員確認済みに変更
# --------------------------------------------------

note_review_records.loc[
    target_mask,
    "ai_note_draft_status"
] = "reviewed"

note_review_records.loc[
    target_mask,
    "ai_note_reviewed_by"
] = "サ責A"

note_review_records.loc[
    target_mask,
    "ai_note_reviewed_at"
] = "2026-06-10 17:45"

print("✅ 特記事項補填案を職員確認済みに変更しました。")
print("状態: reviewed")

display(
    note_review_records.loc[
        target_mask,
        [
            "record_id",
            "user_id",
            "record_date",
            "special_notes",
            "ai_note_draft_status",
            "ai_note_reviewed_by",
            "ai_note_reviewed_at",
            "ai_note_draft",
        ]
    ]
)

✅ 特記事項補填案入りCSVを読み込みました。
読み込み元: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/daily_records_with_ai_note_draft.csv
✅ 特記事項補填案を職員確認済みに変更しました。
状態: reviewed


,record_id,user_id,record_date,special_notes,ai_note_draft_status,ai_note_reviewed_by,ai_note_reviewed_at,ai_note_draft
0,R001,U001,2026-06-06,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,reviewed,サ責A,2026-06-10 17:45,【特記事項の補填案】\n居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふら...


In [24]:
# --------------------------------------------------
# 4. 職員確認済みCSVを別名保存
# --------------------------------------------------

note_reviewed_output_path = BASE_DIR / "daily_records_note_reviewed.csv"

note_review_records.to_csv(
    note_reviewed_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ 特記事項補填案の職員確認済みCSVを保存しました。")
print("保存先:", note_reviewed_output_path)

✅ 特記事項補填案の職員確認済みCSVを保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/daily_records_note_reviewed.csv


In [25]:
# --------------------------------------------------
# 5. 保存結果を確認
# --------------------------------------------------

if not note_reviewed_output_path.exists():
    raise FileNotFoundError(
        "daily_records_note_reviewed.csv が見つかりません。"
    )

saved_note_reviewed = pd.read_csv(note_reviewed_output_path)

target_row_df = saved_note_reviewed[
    saved_note_reviewed["record_id"] == target_record_id
]

if len(target_row_df) != 1:
    raise ValueError(
        f"{target_record_id} の確認済み記録が1件ではありません。"
    )

target_row = target_row_df.iloc[0]

checks = {
    "対象record_idが存在する": pd.notna(target_row["record_id"]),
    "AI補填案が保存されている": (
        pd.notna(target_row["ai_note_draft"])
        and str(target_row["ai_note_draft"]).strip() != ""
    ),
    "職員確認済みになっている": (
        target_row["ai_note_draft_status"] == "reviewed"
    ),
    "確認者が記録されている": (
        pd.notna(target_row["ai_note_reviewed_by"])
        and str(target_row["ai_note_reviewed_by"]).strip() != ""
    ),
    "確認日時が記録されている": (
        pd.notna(target_row["ai_note_reviewed_at"])
        and str(target_row["ai_note_reviewed_at"]).strip() != ""
    ),
}

print("=== 特記事項補填案 職員確認済みチェック ===")

for label, result in checks.items():
    mark = "✅" if result else "⚠️"
    print(f"{mark} {label}")

if all(checks.values()):
    print("\n✅ 特記事項補填案の生成から職員確認済み保存まで正常に完了しました。")
else:
    print("\n⚠️ 確認が必要な項目があります。")

display(
    target_row_df[
        [
            "record_id",
            "user_id",
            "record_date",
            "special_notes",
            "ai_note_draft_status",
            "ai_note_reviewed_by",
            "ai_note_reviewed_at",
            "ai_note_draft",
        ]
    ]
)

=== 特記事項補填案 職員確認済みチェック ===
✅ 対象record_idが存在する
✅ AI補填案が保存されている
✅ 職員確認済みになっている
✅ 確認者が記録されている
✅ 確認日時が記録されている

✅ 特記事項補填案の生成から職員確認済み保存まで正常に完了しました。


,record_id,user_id,record_date,special_notes,ai_note_draft_status,ai_note_reviewed_by,ai_note_reviewed_at,ai_note_draft
0,R001,U001,2026-06-06,居室からトイレまで歩行器を使用して移動。立ち上がり時に軽度のふらつきがあったため、側方より見...,reviewed,サ責A,2026-06-10 17:45,【特記事項の補填案】\n居室からトイレまで歩行器を使用して移動した。立ち上がり時に軽度のふら...


### 12. 月間モニタリングPayload作成

In [26]:
def build_monthly_monitoring_payload(
    user_id: str,
    target_month: str,
) -> dict:
    """
    利用者1名について、指定月の日次記録を集計し、
    月間モニタリング案の生成に使うJSONを作成する。

    Parameters
    ----------
    user_id:
        利用者ID。例: "U004"

    target_month:
        対象月。例: "2026-06"

    Returns
    -------
    dict
        AIへ渡すJSON形式の辞書
    """

    # -----------------------------
    # 1. 対象期間
    # -----------------------------
    period = pd.Period(target_month, freq="M")

    start_date = period.start_time.normalize()
    end_date = period.end_time.normalize()

    # -----------------------------
    # 2. 利用者の存在確認
    # -----------------------------
    user_rows = users[
        users["user_id"] == user_id
    ]

    if user_rows.empty:
        raise ValueError(
            f"利用者ID {user_id} は users.csv に存在しません。"
        )

    # -----------------------------
    # 3. 対象期間に有効な計画書を取得
    # -----------------------------
    user_plans = care_plans[
        care_plans["user_id"] == user_id
    ].copy()

    active_plans = user_plans[
        (user_plans["valid_from"] <= end_date)
        & (user_plans["valid_to"] >= start_date)
    ].sort_values(
        "valid_from",
        ascending=False
    )

    plan_selection_note = "対象期間内の有効な計画書を使用"

    if active_plans.empty:
        # 期限切れでも、直近の計画書を参考資料として取得する。
        # AIには期限切れであることを明示する。
        fallback_plans = user_plans[
            user_plans["valid_from"] <= end_date
        ].sort_values(
            "valid_from",
            ascending=False
        )

        if fallback_plans.empty:
            selected_plan = None
            plan_selection_note = (
                "参照可能な計画書が存在しない。"
                "職員による確認が必要。"
            )

        else:
            selected_plan = fallback_plans.iloc[0]
            plan_selection_note = (
                "対象期間内の有効な計画書が存在しない。"
                "直近の計画書を参考資料として使用。"
                "期限切れの可能性があるため、職員確認が必要。"
            )

    else:
        selected_plan = active_plans.iloc[0]

    # -----------------------------
    # 4. 対象月の日次記録を抽出
    # -----------------------------
    monthly_records = daily_records[
        (daily_records["user_id"] == user_id)
        & (daily_records["record_date"] >= start_date)
        & (daily_records["record_date"] <= end_date)
    ].copy()

    monthly_records = monthly_records.sort_values(
        "record_date"
    )

    monthly_records["notes_length"] = (
        monthly_records["special_notes"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    # 参照した記録の日付範囲

    if monthly_records.empty:
        first_record_date = ""
        last_record_date = ""

    else:
        first_record_date = monthly_records["record_date"].min().strftime("%Y-%m-%d")
        last_record_date = monthly_records["record_date"].max().strftime("%Y-%m-%d")

    # -----------------------------
    # 5. 面接情報を取得
    # -----------------------------
    user_monitoring = monitoring_records[
        monitoring_records["user_id"] == user_id
    ].copy()

    user_monitoring = user_monitoring[
        user_monitoring["record_date"] <= end_date
    ].sort_values(
        "record_date",
        ascending=False
    )

    if user_monitoring.empty:
        monitoring_context = {
            "interview_date": "",
            "interview_method": "",
            "attendees": [],
            "information_sources": [],
            "context_note": (
                "面接情報が未入力です。"
                "職員による確認が必要です。"
            ),
        }

    else:
        latest_monitoring = user_monitoring.iloc[0]

        interview_date = latest_monitoring[
            "interview_date"
        ]

        monitoring_context = {
            "interview_date": (
                ""
                if pd.isna(interview_date)
                else interview_date.strftime("%Y-%m-%d")
            ),
            "interview_method": clean_text(
                latest_monitoring["interview_method"]
            ),
            "attendees": split_items(
                latest_monitoring["attendees"]
            ),
            "information_sources": split_items(
                latest_monitoring["information_sources"]
            ),
            "context_note": (
                "面接日、方法、同席者、情報源は、"
                "AIが推測せず職員入力を使用する。"
            ),
        }

    # -----------------------------
    # 6. 日次記録をJSON向けに整理
    # -----------------------------
    record_items = []

    for _, row in monthly_records.iterrows():
        record_items.append(
            {
                "record_id": clean_text(
                    row["record_id"]
                ),
                "record_date": row[
                    "record_date"
                ].strftime("%Y-%m-%d"),
                "service_items": split_items(
                    row["service_items"]
                ),
                "special_notes": clean_text(
                    row["special_notes"]
                ),
                "notes_length": int(
                    row["notes_length"]
                ),
                "user_quote": clean_text(
                    row["user_quote"]
                ),
                "observation": clean_text(
                    row["observation"]
                ),
                "numeric_data": clean_text(
                    row["numeric_data"]
                ),
                "follow_up": clean_text(
                    row["follow_up"]
                ),
                "approved": bool(
                    row["approved"]
                ),
            }
        )

    # -----------------------------
    # 7. 計画書をJSON向けに整理
    # -----------------------------
    if selected_plan is None:
        care_plan_payload = {
            "plan_id": "",
            "valid_from": "",
            "valid_to": "",
            "goal": "",
            "service_content": "",
            "precautions": "",
            "selection_note": plan_selection_note,
        }

    else:
        care_plan_payload = {
            "plan_id": clean_text(
                selected_plan["plan_id"]
            ),
            "valid_from": (
                selected_plan["valid_from"]
                .strftime("%Y-%m-%d")
            ),
            "valid_to": (
                selected_plan["valid_to"]
                .strftime("%Y-%m-%d")
            ),
            "goal": clean_text(
                selected_plan["goal"]
            ),
            "service_content": clean_text(
                selected_plan["service_content"]
            ),
            "precautions": clean_text(
                selected_plan["precautions"]
            ),
            "selection_note": plan_selection_note,
        }

    # -----------------------------
    # 8. AIへ渡すJSON
    # -----------------------------
    payload = {
        "task": "generate_monthly_monitoring_draft",
        "user_id": user_id,
        "target_period": {
            "start_date": start_date.strftime("%Y-%m-%d"),
            "end_date": end_date.strftime("%Y-%m-%d"),
        },
        "care_plan": care_plan_payload,
        "monitoring_context": monitoring_context,
        "daily_records_summary": {
            "record_count": int(
                len(monthly_records)
            ),
            "first_record_date": first_record_date,
            "last_record_date": last_record_date,
            "short_record_count": int(
                (
                    monthly_records["notes_length"]
                    < 100
                ).sum()
            ),
            "unapproved_record_count": int(
                (
                    monthly_records["approved"]
                    == False
                ).sum()
            ),
            "records": record_items,
        },
        "generation_rules": [
            (
                "日次記録、計画書、職員が入力した面接情報に"
                "存在する事実だけを使用する。"
            ),
            (
                "存在しない発言、数値、支援、面接、同席者、"
                "署名、同意を推測して追加しない。"
            ),
            (
                "本人の発言を引用する場合は、"
                "user_quote に記載された内容だけを使用する。"
            ),
            (
                "日次記録の傾向と計画目標を比較し、"
                "達成状況の評価案を作る。"
            ),
            (
                "不明な事項は推測せず、"
                "職員確認が必要と明示する。"
            ),
            (
                "過去日付で作成済みであったように"
                "見せる文章を生成しない。"
            ),
            (
                "法令違反、行政処分、返還の有無を"
                "断定しない。"
            ),
        ],
        "required_output": {
            "language": "Japanese",
            "draft_length": "300〜500文字程度",
            "required_sections": [
                "実施の事実",
                "客観的な事実",
                "計画目標に対する評価案",
                "今後の対応案",
                "職員確認が必要な事項",
            ],
            "human_review_required": True,
        },
    }

    return payload

###13. 月間モニタリング生成プロンプト作成

### 13-1. payloadの作成・確認

In [27]:
payload = build_monthly_monitoring_payload(
    user_id="U004",
    target_month="2026-06",
)

print(
    json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "task": "generate_monthly_monitoring_draft",
  "user_id": "U004",
  "target_period": {
    "start_date": "2026-06-01",
    "end_date": "2026-06-30"
  },
  "care_plan": {
    "plan_id": "P004",
    "valid_from": "2026-05-01",
    "valid_to": "2026-10-31",
    "goal": "食事摂取量を維持し、体調を安定させる",
    "service_content": "食事見守り、水分摂取の声かけ、体調観察",
    "precautions": "食欲低下、発熱、脱水兆候に注意",
    "selection_note": "対象期間内の有効な計画書を使用"
  },
  "monitoring_context": {
    "interview_date": "2026-06-10",
    "interview_method": "訪問",
    "attendees": [
      "本人"
    ],
    "information_sources": [
      "本人",
      "日次記録",
      "看護職申し送り"
    ],
    "context_note": "面接日、方法、同席者、情報源は、AIが推測せず職員入力を使用する。"
  },
  "daily_records_summary": {
    "record_count": 4,
    "first_record_date": "2026-06-04",
    "last_record_date": "2026-06-10",
    "short_record_count": 4,
    "unapproved_record_count": 0,
    "records": [
      {
        "record_id": "R009",
        "record_date": "2026-06-04",
        "service_items": [


### 13-2. プロンプトの作成

In [28]:
# --------------------------------------------------
# AIへ渡す月間モニタリング生成用プロンプトを作成
# --------------------------------------------------

care_plan = payload["care_plan"]
records = payload["daily_records_summary"]["records"]

records_text = "\n".join(
    [
        f"""
【記録日】{record["record_date"]}
【サービス内容】{record["service_items"]}
【特記事項】{record["special_notes"]}
【利用者の発言】{record["user_quote"]}
【観察事項】{record["observation"]}
【今後の対応】{record["follow_up"]}
""".strip()
        for record in records
    ]
)

monitoring_prompt = f"""
あなたは訪問介護事業所のサービス提供責任者を支援するAIです。
以下のケアプランと日々の介護記録をもとに、
対象期間の月間モニタリング文書の下書きを作成してください。

【重要事項】
- 事実として記録されていない内容を推測で追加しないでください。
- 医学的な診断を行わないでください。
- 下書きは必ず職員が確認・修正する前提で作成してください。
- 状態変化がある場合は、根拠となる記録日を明記してください。
- 情報が不足している場合は、不足事項として明記してください。

【利用者ID】
{payload["user_id"]}

【対象期間】
{payload["target_period"]["start_date"]} ～ {payload["target_period"]["end_date"]}

【参照した記録期間】
{payload["daily_records_summary"]["first_record_date"]} ～ {payload["daily_records_summary"]["last_record_date"]}

【ケアプラン】
目標：{care_plan["goal"]}
支援内容：{care_plan["service_content"]}
注意事項：{care_plan["precautions"]}

【日々の記録】
{records_text}

【出力項目】
1. 総合評価
2. ケアプラン目標に対する評価
3. 状態変化と根拠
4. 今後の対応案
5. 職員が確認すべき事項
6. 情報不足の項目
""".strip()

print(monitoring_prompt)

あなたは訪問介護事業所のサービス提供責任者を支援するAIです。
以下のケアプランと日々の介護記録をもとに、
対象期間の月間モニタリング文書の下書きを作成してください。

【重要事項】
- 事実として記録されていない内容を推測で追加しないでください。
- 医学的な診断を行わないでください。
- 下書きは必ず職員が確認・修正する前提で作成してください。
- 状態変化がある場合は、根拠となる記録日を明記してください。
- 情報が不足している場合は、不足事項として明記してください。

【利用者ID】
U004

【対象期間】
2026-06-01 ～ 2026-06-30

【参照した記録期間】
2026-06-04 ～ 2026-06-10

【ケアプラン】
目標：食事摂取量を維持し、体調を安定させる
支援内容：食事見守り、水分摂取の声かけ、体調観察
注意事項：食欲低下、発熱、脱水兆候に注意

【日々の記録】
【記録日】2026-06-04
【サービス内容】['食事見守り', '水分摂取']
【特記事項】昼食は主食5割、副食4割の摂取であった。普段より食欲が低下している様子。水分は声かけによりコップ1杯摂取された。
【利用者の発言】あまり食べたくない
【観察事項】食欲低下あり。水分は声かけで摂取。
【今後の対応】食事量と水分量を継続観察する
【記録日】2026-06-06
【サービス内容】['食事見守り', '体調観察']
【特記事項】朝食は半量程度で終了。本人より食欲が出ないとの訴えあり。倦怠感がみられたため、看護職へ申し送りを行った。
【利用者の発言】食欲が出ない
【観察事項】倦怠感あり。看護職へ申し送り。
【今後の対応】看護職と共有し継続観察する
【記録日】2026-06-08
【サービス内容】['食事見守り', '水分摂取']
【特記事項】昼食摂取量は3割程度。水分摂取も少なく、複数回の声かけを行った。食欲低下が継続しているため、継続観察が必要。
【利用者の発言】今日はあまり食べられない
【観察事項】食欲低下が継続。水分摂取も少ない。
【今後の対応】看護職へ共有し、食事量と水分量を確認する
【記録日】2026-06-10
【サービス内容】['体調観察']
【特記事項】食事量低下が継続。本人より少しだるいとの発言あり。発熱は認めないが、水分摂取量も少ないため看護職へ

### 14. AI下書き保存

In [29]:
# --------------------------------------------------
# AI生成下書きを monitoring_records に一時保存する
# --------------------------------------------------

target_user_id = "U004"

ai_generated_draft = """
## 月間モニタリング文書（下書き）

**利用者ID：** U004
**対象期間：** 2026年6月1日 ～ 2026年6月30日
**参照した記録期間：** 2026年6月4日 ～ 2026年6月10日

### 1. 総合評価

参照した記録期間において、食事摂取量の低下が継続して確認された。
2026年6月4日は昼食の主食5割、副食4割、6月6日は朝食半量程度、6月8日は昼食3割程度の摂取であった。本人からも「あまり食べたくない」「食欲が出ない」「今日はあまり食べられない」との発言があった。

水分摂取については、6月4日は声かけによりコップ1杯の摂取が確認されたが、6月8日および6月10日は摂取量が少ない状態が記録されている。

また、6月6日に倦怠感がみられ、6月10日には本人から「少しだるい」との発言があった。6月10日時点では発熱は認められていない。食事量および水分摂取量の低下が継続しているため、看護職への共有が行われている。

### 2. ケアプラン目標に対する評価

ケアプランの目標は「食事摂取量を維持し、体調を安定させる」である。

参照した記録期間では、食事摂取量の低下が継続しており、目標を達成できているとは判断しにくい状況である。水分摂取量についても低下が記録されており、声かけおよび継続的な観察が必要である。

体調面では、倦怠感および本人からのだるさの訴えが確認されている。一方、6月10日時点では発熱は認められていない。

### 3. 状態変化と根拠

#### 食事摂取量の低下

* **2026年6月4日：** 昼食は主食5割、副食4割。本人より「あまり食べたくない」との発言あり。
* **2026年6月6日：** 朝食は半量程度。本人より「食欲が出ない」との発言あり。
* **2026年6月8日：** 昼食は3割程度。本人より「今日はあまり食べられない」との発言あり。
* **2026年6月10日：** 食事量低下が継続していると記録されている。

#### 水分摂取量の低下

* **2026年6月4日：** 声かけによりコップ1杯の摂取あり。
* **2026年6月8日：** 水分摂取量が少なく、複数回の声かけを実施。
* **2026年6月10日：** 水分摂取量の低下が継続していると記録されている。

#### 体調に関する変化

* **2026年6月6日：** 倦怠感がみられたため、看護職へ申し送りを実施。
* **2026年6月10日：** 本人より「少しだるい」との発言あり。発熱は認められていない。看護職へ再度共有している。

### 4. 今後の対応案

* 食事摂取量および水分摂取量の観察を継続する。
* 食事および水分摂取時の声かけを継続する。
* 倦怠感、だるさの訴え、発熱の有無など、体調の変化を継続して確認する。
* 看護職と情報共有を継続する。
* 食事量および水分摂取量の低下が継続しているため、ケアプラン見直しの要否を職員間で確認する。

### 5. 職員が確認すべき事項

* 看護職への申し送り後の確認結果および指示内容
* 食事摂取量の低下が継続している期間
* 水分摂取量の具体的な記録方法
* 6月10日以降の食事量、水分量および体調の変化
* ケアプラン見直しの要否
* 必要に応じた関係職種への情報共有状況

### 6. 情報不足の項目

以下の情報は、参照した記録からは確認できない。

* 2026年6月1日から6月3日までの記録
* 2026年6月11日から6月30日までの記録
* 水分摂取量の具体的な総量
* 普段の食事摂取量および水分摂取量との比較
* 体温の具体的な測定値
* 看護職への申し送り後の判断および対応内容
* ケアプラン見直しの実施有無
* 体重変化の有無
* 排泄状況など、体調確認に関する追加情報

※本書は月間モニタリング文書の下書きである。サービス提供責任者および関係職員が内容を確認し、必要に応じて修正・追記すること。
""".strip()

# 対象利用者の行を確認
target_mask = monitoring_records["user_id"] == target_user_id

if target_mask.sum() == 0:
    raise ValueError(f"{target_user_id} のモニタリング記録が見つかりません。")

if target_mask.sum() > 1:
    raise ValueError(
        f"{target_user_id} のモニタリング記録が複数あります。"
        "対象月を追加条件として指定してください。"
    )

# AI下書き用の列を文字列として扱う
# 空欄列を持つCSVを読み込んだ場合、pandasがfloat型として解釈し、
# 長文テキスト代入時に FutureWarning が出ることを防ぐ。
for col in ["ai_draft", "ai_draft_status"]:
    if col not in monitoring_records.columns:
        monitoring_records[col] = ""
    monitoring_records[col] = monitoring_records[col].astype("object")

# 正式記録は上書きせず、AI下書き用の列だけ更新
monitoring_records.loc[target_mask, "ai_draft"] = ai_generated_draft
monitoring_records.loc[target_mask, "ai_draft_status"] = "generated_pending_review"

print("✅ AI生成下書きを一時保存しました。")
print("状態: generated_pending_review")

✅ AI生成下書きを一時保存しました。
状態: generated_pending_review


In [30]:
display(
    monitoring_records.loc[
        monitoring_records["user_id"] == "U004",
        [
            "monitoring_id",
            "user_id",
            "record_date",
            "approved",
            "ai_draft_status",
            "ai_draft"
        ]
    ]
)

,monitoring_id,user_id,record_date,approved,ai_draft_status,ai_draft
3,M004,U004,2026-06-10,True,generated_pending_review,## 月間モニタリング文書（下書き）\n\n**利用者ID：** U004\n**対象期間：...


In [31]:
output_path = BASE_DIR / "monitoring_records_with_ai_draft.csv"

monitoring_records.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ AI下書き入りCSVを別名で保存しました。")
print("保存先:", output_path)

✅ AI下書き入りCSVを別名で保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/monitoring_records_with_ai_draft.csv


### 15. 職員確認済み保存

In [32]:
# --------------------------------------------------
# AI下書き入りCSVを読み込む
# --------------------------------------------------

review_source_path = BASE_DIR / "monitoring_records_with_ai_draft.csv"

review_records = pd.read_csv(review_source_path)

# --------------------------------------------------
# U004のAI下書きを職員確認済みにする
# --------------------------------------------------

target_user_id = "U004"
target_mask = review_records["user_id"] == target_user_id

if target_mask.sum() == 0:
    raise ValueError(f"{target_user_id} の記録が見つかりません。")

if target_mask.sum() > 1:
    raise ValueError(
        f"{target_user_id} の記録が複数あります。"
        "対象月を追加条件として指定してください。"
    )

# AI下書きの確認状態を変更
review_records.loc[target_mask, "ai_draft_status"] = "reviewed"

# 誰が、いつ確認したかを記録する列を追加
review_records.loc[target_mask, "ai_reviewed_by"] = "サ責A"
review_records.loc[target_mask, "ai_reviewed_at"] = "2026-06-10 17:00"

print("✅ U004のAI下書きを職員確認済みに変更しました。")

✅ U004のAI下書きを職員確認済みに変更しました。


In [33]:
display(
    review_records.loc[
        review_records["user_id"] == "U004",
        [
            "monitoring_id",
            "user_id",
            "approved",
            "ai_draft_status",
            "ai_reviewed_by",
            "ai_reviewed_at"
        ]
    ]
)

,monitoring_id,user_id,approved,ai_draft_status,ai_reviewed_by,ai_reviewed_at
3,M004,U004,True,reviewed,サ責A,2026-06-10 17:00


In [34]:
reviewed_output_path = BASE_DIR / "monitoring_records_reviewed.csv"

review_records.to_csv(
    reviewed_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("✅ 職員確認済みCSVを別名で保存しました。")
print("保存先:", reviewed_output_path)

✅ 職員確認済みCSVを別名で保存しました。
保存先: /content/drive/MyDrive/Care_Compliance_Copilot/02_擬似データ/monitoring_records_reviewed.csv


### 16. 全体最終チェック

In [35]:
# care_plans.csv と document_status.csv の
# 訪問介護計画書の有効期間が一致しているか確認する

plan_status = document_status[
    document_status["document_type"] == "訪問介護計画書"
][
    ["user_id", "valid_from", "valid_to"]
].copy()

comparison = care_plans[
    ["user_id", "valid_from", "valid_to"]
].merge(
    plan_status,
    on="user_id",
    how="left",
    suffixes=("_care_plan", "_document_status")
)

comparison["valid_from_match"] = (
    comparison["valid_from_care_plan"]
    == comparison["valid_from_document_status"]
)

comparison["valid_to_match"] = (
    comparison["valid_to_care_plan"]
    == comparison["valid_to_document_status"]
)

display(comparison)

if comparison[["valid_from_match", "valid_to_match"]].all().all():
    print("✅ care_plans.csv と document_status.csv の有効期間は一致しています。")
else:
    print("⚠️ 有効期間が一致していない利用者があります。")

,user_id,valid_from_care_plan,valid_to_care_plan,valid_from_document_status,valid_to_document_status,valid_from_match,valid_to_match
0,U001,2026-06-01,2026-11-30,2026-06-01,2026-11-30,True,True
1,U002,2026-04-01,2026-09-30,2026-04-01,2026-09-30,True,True
2,U003,2026-03-01,2026-08-31,2026-03-01,2026-08-31,True,True
3,U004,2026-05-01,2026-10-31,2026-05-01,2026-10-31,True,True
4,U005,2026-01-01,2026-06-30,2026-01-01,2026-06-30,True,True


✅ care_plans.csv と document_status.csv の有効期間は一致しています。


In [36]:
# --------------------------------------------------
# 確認済みCSVの最終チェック
# --------------------------------------------------

reviewed_path = BASE_DIR / "monitoring_records_reviewed.csv"

if not reviewed_path.exists():
    raise FileNotFoundError(
        "monitoring_records_reviewed.csv が見つかりません。"
    )

final_records = pd.read_csv(reviewed_path)

print("確認済みCSVの行数:", len(final_records))
print("確認済みCSVの列数:", len(final_records.columns))

u004 = final_records[
    final_records["user_id"] == "U004"
]

if len(u004) != 1:
    raise ValueError(
        "U004の記録が1件ではありません。"
    )

u004_row = u004.iloc[0]

checks = {
    "U004の記録が存在する": len(u004) == 1,
    "AI下書きが保存されている": (
        pd.notna(u004_row["ai_draft"])
        and str(u004_row["ai_draft"]).strip() != ""
    ),
    "職員確認済みになっている": (
        u004_row["ai_draft_status"] == "reviewed"
    ),
    "確認者が記録されている": (
        pd.notna(u004_row["ai_reviewed_by"])
        and str(u004_row["ai_reviewed_by"]).strip() != ""
    ),
    "確認日時が記録されている": (
        pd.notna(u004_row["ai_reviewed_at"])
        and str(u004_row["ai_reviewed_at"]).strip() != ""
    )
}

print("\n最終チェック結果:")
for label, result in checks.items():
    mark = "✅" if result else "⚠️"
    print(f"{mark} {label}")

if all(checks.values()):
    print("\n✅ AI下書き生成から職員確認済み保存まで正常に完了しました。")
else:
    print("\n⚠️ 確認が必要な項目があります。")

確認済みCSVの行数: 5
確認済みCSVの列数: 20

最終チェック結果:
✅ U004の記録が存在する
✅ AI下書きが保存されている
✅ 職員確認済みになっている
✅ 確認者が記録されている
✅ 確認日時が記録されている

✅ AI下書き生成から職員確認済み保存まで正常に完了しました。


In [37]:
# --------------------------------------------------
# 1. チェック対象ファイルのパス
# --------------------------------------------------

alerts_integrated_path = BASE_DIR / "alerts_integrated.csv"
alert_summary_path = BASE_DIR / "alert_summary_by_user.csv"
note_reviewed_path = BASE_DIR / "daily_records_note_reviewed.csv"
monitoring_reviewed_path = BASE_DIR / "monitoring_records_reviewed.csv"

# 今回の検証対象
TARGET_NOTE_RECORD_ID = "R001"
TARGET_MONITORING_USER_ID = "U004"


# --------------------------------------------------
# 2. 共通チェック関数
# --------------------------------------------------

def print_check(label: str, result: bool):
    mark = "✅" if result else "⚠️"
    print(f"{mark} {label}")


overall_checks = {}


# --------------------------------------------------
# 3. alerts_integrated.csv の確認
# --------------------------------------------------

print("=== 1. 統合アラート一覧 alerts_integrated.csv の確認 ===")

alerts_file_exists = alerts_integrated_path.exists()
overall_checks["alerts_integrated.csv が存在する"] = alerts_file_exists

print_check(
    "alerts_integrated.csv が存在する",
    alerts_file_exists
)

if alerts_file_exists:
    final_alerts = pd.read_csv(alerts_integrated_path)

    required_alert_columns = [
        "user_id",
        "alert_category",
        "alert_type",
        "severity",
        "document_type",
        "target_type",
        "target_id",
        "detail",
        "action_type",
        "human_review_required",
    ]

    alerts_has_rows = len(final_alerts) > 0
    alerts_has_required_columns = all(
        col in final_alerts.columns
        for col in required_alert_columns
    )
    alerts_has_action_type = (
        "action_type" in final_alerts.columns
        and final_alerts["action_type"].notna().any()
    )

    overall_checks["alerts_integrated.csv に行が存在する"] = alerts_has_rows
    overall_checks["alerts_integrated.csv に必要列が存在する"] = alerts_has_required_columns
    overall_checks["action_type が設定されている"] = alerts_has_action_type

    print("行数:", len(final_alerts))
    print("列数:", len(final_alerts.columns))

    print_check(
        "alerts_integrated.csv に行が存在する",
        alerts_has_rows
    )
    print_check(
        "alerts_integrated.csv に必要列が存在する",
        alerts_has_required_columns
    )
    print_check(
        "action_type が設定されている",
        alerts_has_action_type
    )

    display(final_alerts.head())


# --------------------------------------------------
# 4. 利用者別アラート一覧 alert_summary_by_user.csv の確認
# --------------------------------------------------

print("\n=== 2. 利用者別アラート一覧 alert_summary_by_user.csv の確認 ===")

summary_file_exists = alert_summary_path.exists()
overall_checks["alert_summary_by_user.csv が存在する"] = summary_file_exists

print_check(
    "alert_summary_by_user.csv が存在する",
    summary_file_exists
)

if summary_file_exists:
    final_alert_summary = pd.read_csv(alert_summary_path)

    required_summary_columns = [
        "user_id",
        "user_name",
        "alert_count",
        "uncreated_count",
        "unapproved_count",
        "unconfirmed_count",
        "uncreated_documents",
        "unapproved_documents",
        "unconfirmed_documents",
        "uncreated_targets_json",
        "unapproved_targets_json",
        "unconfirmed_targets_json",
        "alert_message",
    ]

    summary_has_rows = len(final_alert_summary) > 0
    summary_has_required_columns = all(
        col in final_alert_summary.columns
        for col in required_summary_columns
    )
    summary_has_alert_message = (
        "alert_message" in final_alert_summary.columns
        and final_alert_summary["alert_message"].notna().any()
    )

    overall_checks["alert_summary_by_user.csv に行が存在する"] = summary_has_rows
    overall_checks["状態別の書類名列が存在する"] = summary_has_required_columns
    overall_checks["アラート文章が作成されている"] = summary_has_alert_message

    print("行数:", len(final_alert_summary))
    print("列数:", len(final_alert_summary.columns))

    print_check(
        "alert_summary_by_user.csv に行が存在する",
        summary_has_rows
    )
    print_check(
        "状態別の書類名列が存在する",
        summary_has_required_columns
    )
    print_check(
        "アラート文章が作成されている",
        summary_has_alert_message
    )

    display(final_alert_summary.head())


# --------------------------------------------------
# 5. 特記事項補填案の職員確認済みCSVの確認
# --------------------------------------------------

print("\n=== 3. 特記事項補填案 daily_records_note_reviewed.csv の確認 ===")

note_file_exists = note_reviewed_path.exists()
overall_checks["daily_records_note_reviewed.csv が存在する"] = note_file_exists

print_check(
    "daily_records_note_reviewed.csv が存在する",
    note_file_exists
)

if note_file_exists:
    final_note_records = pd.read_csv(note_reviewed_path)

    target_note_df = final_note_records[
        final_note_records["record_id"] == TARGET_NOTE_RECORD_ID
    ]

    note_target_exists = len(target_note_df) == 1

    overall_checks["対象の特記事項記録が1件存在する"] = note_target_exists

    print("行数:", len(final_note_records))
    print("列数:", len(final_note_records.columns))

    print_check(
        f"{TARGET_NOTE_RECORD_ID} の記録が1件存在する",
        note_target_exists
    )

    if note_target_exists:
        note_row = target_note_df.iloc[0]

        note_draft_exists = (
            pd.notna(note_row.get("ai_note_draft"))
            and str(note_row.get("ai_note_draft")).strip() != ""
        )
        note_status_reviewed = (
            note_row.get("ai_note_draft_status") == "reviewed"
        )
        note_reviewed_by_exists = (
            pd.notna(note_row.get("ai_note_reviewed_by"))
            and str(note_row.get("ai_note_reviewed_by")).strip() != ""
        )
        note_reviewed_at_exists = (
            pd.notna(note_row.get("ai_note_reviewed_at"))
            and str(note_row.get("ai_note_reviewed_at")).strip() != ""
        )

        overall_checks["特記事項AI補填案が保存されている"] = note_draft_exists
        overall_checks["特記事項AI補填案が職員確認済み"] = note_status_reviewed
        overall_checks["特記事項AI補填案の確認者が記録されている"] = note_reviewed_by_exists
        overall_checks["特記事項AI補填案の確認日時が記録されている"] = note_reviewed_at_exists

        print_check(
            "AI補填案が保存されている",
            note_draft_exists
        )
        print_check(
            "AI補填案が職員確認済みになっている",
            note_status_reviewed
        )
        print_check(
            "確認者が記録されている",
            note_reviewed_by_exists
        )
        print_check(
            "確認日時が記録されている",
            note_reviewed_at_exists
        )

        display(
            target_note_df[
                [
                    "record_id",
                    "user_id",
                    "record_date",
                    "special_notes",
                    "ai_note_draft_status",
                    "ai_note_reviewed_by",
                    "ai_note_reviewed_at",
                    "ai_note_draft",
                ]
            ]
        )


# --------------------------------------------------
# 6. 月間モニタリング職員確認済みCSVの確認
# --------------------------------------------------

print("\n=== 4. 月間モニタリング monitoring_records_reviewed.csv の確認 ===")

monitoring_file_exists = monitoring_reviewed_path.exists()
overall_checks["monitoring_records_reviewed.csv が存在する"] = monitoring_file_exists

print_check(
    "monitoring_records_reviewed.csv が存在する",
    monitoring_file_exists
)

if monitoring_file_exists:
    final_monitoring_records = pd.read_csv(monitoring_reviewed_path)

    target_monitoring_df = final_monitoring_records[
        final_monitoring_records["user_id"] == TARGET_MONITORING_USER_ID
    ]

    monitoring_target_exists = len(target_monitoring_df) == 1

    overall_checks["対象利用者のモニタリング記録が1件存在する"] = monitoring_target_exists

    print("行数:", len(final_monitoring_records))
    print("列数:", len(final_monitoring_records.columns))

    print_check(
        f"{TARGET_MONITORING_USER_ID} のモニタリング記録が1件存在する",
        monitoring_target_exists
    )

    if monitoring_target_exists:
        monitoring_row = target_monitoring_df.iloc[0]

        monitoring_draft_exists = (
            pd.notna(monitoring_row.get("ai_draft"))
            and str(monitoring_row.get("ai_draft")).strip() != ""
        )
        monitoring_status_reviewed = (
            monitoring_row.get("ai_draft_status") == "reviewed"
        )
        monitoring_reviewed_by_exists = (
            pd.notna(monitoring_row.get("ai_reviewed_by"))
            and str(monitoring_row.get("ai_reviewed_by")).strip() != ""
        )
        monitoring_reviewed_at_exists = (
            pd.notna(monitoring_row.get("ai_reviewed_at"))
            and str(monitoring_row.get("ai_reviewed_at")).strip() != ""
        )

        overall_checks["月間モニタリングAI下書きが保存されている"] = monitoring_draft_exists
        overall_checks["月間モニタリングAI下書きが職員確認済み"] = monitoring_status_reviewed
        overall_checks["月間モニタリング確認者が記録されている"] = monitoring_reviewed_by_exists
        overall_checks["月間モニタリング確認日時が記録されている"] = monitoring_reviewed_at_exists

        print_check(
            "AI下書きが保存されている",
            monitoring_draft_exists
        )
        print_check(
            "AI下書きが職員確認済みになっている",
            monitoring_status_reviewed
        )
        print_check(
            "確認者が記録されている",
            monitoring_reviewed_by_exists
        )
        print_check(
            "確認日時が記録されている",
            monitoring_reviewed_at_exists
        )

        display(
            target_monitoring_df[
                [
                    "monitoring_id",
                    "user_id",
                    "record_date",
                    "approved",
                    "ai_draft_status",
                    "ai_reviewed_by",
                    "ai_reviewed_at",
                    "ai_draft",
                ]
            ]
        )


# --------------------------------------------------
# 7. care_plans.csv と document_status.csv の整合性確認
# --------------------------------------------------

print("\n=== 5. 計画書有効期間の整合性確認 ===")

plan_status = document_status[
    document_status["document_type"] == "訪問介護計画書"
][
    ["user_id", "valid_from", "valid_to"]
].copy()

comparison = care_plans[
    ["user_id", "valid_from", "valid_to"]
].merge(
    plan_status,
    on="user_id",
    how="left",
    suffixes=("_care_plan", "_document_status")
)

comparison["valid_from_match"] = (
    comparison["valid_from_care_plan"]
    == comparison["valid_from_document_status"]
)

comparison["valid_to_match"] = (
    comparison["valid_to_care_plan"]
    == comparison["valid_to_document_status"]
)

periods_match = comparison[
    ["valid_from_match", "valid_to_match"]
].all().all()

overall_checks["care_plans と document_status の有効期間が一致している"] = periods_match

print_check(
    "care_plans.csv と document_status.csv の有効期間が一致している",
    periods_match
)

display(comparison)


# --------------------------------------------------
# 8. 全体結果
# --------------------------------------------------

print("\n=== 6. MVP全体チェック結果 ===")

for label, result in overall_checks.items():
    print_check(label, result)

if all(overall_checks.values()):
    print("\n✅ Care Compliance Copilot MVP の主要フローは正常に完了しました。")
    print("✅ 統合アラート検知、状態別アラート表示、特記事項補填案、月間モニタリング下書き、職員確認済み保存まで確認済みです。")
else:
    print("\n⚠️ 一部確認が必要な項目があります。")

=== 1. 統合アラート一覧 alerts_integrated.csv の確認 ===
✅ alerts_integrated.csv が存在する
行数: 17
列数: 10
✅ alerts_integrated.csv に行が存在する
✅ alerts_integrated.csv に必要列が存在する
✅ action_type が設定されている


,user_id,alert_category,alert_type,severity,document_type,target_type,target_id,detail,action_type,human_review_required
0,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R001,特記事項が100文字未満です。現在の文字数: 69文字,generate_note_draft,True
1,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R002,特記事項が100文字未満です。現在の文字数: 56文字,generate_note_draft,True
2,U001,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R003,特記事項が100文字未満です。現在の文字数: 52文字,generate_note_draft,True
3,U002,document,当月モニタリング未確認,high,モニタリング記録,monitoring,D004,2026年6月分のモニタリング記録が確認できません。,generate_monitoring_draft,True
4,U002,daily_record,特記事項不足,medium,日々の介護記録,daily_record,R004,特記事項が100文字未満です。現在の文字数: 54文字,generate_note_draft,True



=== 2. 利用者別アラート一覧 alert_summary_by_user.csv の確認 ===
✅ alert_summary_by_user.csv が存在する
行数: 5
列数: 13
✅ alert_summary_by_user.csv に行が存在する
✅ 状態別の書類名列が存在する
✅ アラート文章が作成されている


,user_id,user_name,alert_count,uncreated_count,unapproved_count,unconfirmed_count,uncreated_documents,unapproved_documents,unconfirmed_documents,uncreated_targets_json,unapproved_targets_json,unconfirmed_targets_json,alert_message
0,U005,利用者E,4,0,1,3,なし,モニタリング記録（1件）,日々の介護記録（3件）,[],"[{""alert_type"": ""未承認"", ""document_type"": ""モニタリン...","[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者E（U005）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
1,U004,利用者D,4,0,0,4,なし,なし,日々の介護記録（4件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者D（U004）には、対応が必要な書類・記録が4件あります。未作成：0件（なし）。未承認...
2,U001,利用者A,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者A（U001）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
3,U002,利用者B,3,0,0,3,なし,なし,日々の介護記録（2件）、モニタリング記録（1件）,[],[],"[{""alert_type"": ""当月モニタリング未確認"", ""document_type""...",利用者B（U002）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...
4,U003,利用者C,3,0,0,3,なし,なし,日々の介護記録（3件）,[],[],"[{""alert_type"": ""特記事項不足"", ""document_type"": ""日々...",利用者C（U003）には、対応が必要な書類・記録が3件あります。未作成：0件（なし）。未承認...



=== 3. 特記事項補填案 daily_records_note_reviewed.csv の確認 ===
✅ daily_records_note_reviewed.csv が存在する
行数: 15
列数: 22
✅ R001 の記録が1件存在する
✅ AI補填案が保存されている
✅ AI補填案が職員確認済みになっている
✅ 確認者が記録されている
✅ 確認日時が記録されている


...


=== 4. 月間モニタリング monitoring_records_reviewed.csv の確認 ===
✅ monitoring_records_reviewed.csv が存在する
行数: 5
列数: 20
✅ U004 のモニタリング記録が1件存在する
✅ AI下書きが保存されている
✅ AI下書きが職員確認済みになっている
✅ 確認者が記録されている
✅ 確認日時が記録されている


,monitoring_id,user_id,record_date,approved,ai_draft_status,ai_reviewed_by,ai_reviewed_at,ai_draft
3,M004,U004,2026-06-10,True,reviewed,サ責A,2026-06-10 17:00,## 月間モニタリング文書（下書き）\n\n**利用者ID：** U004\n**対象期間：...



=== 5. 計画書有効期間の整合性確認 ===
✅ care_plans.csv と document_status.csv の有効期間が一致している


,user_id,valid_from_care_plan,valid_to_care_plan,valid_from_document_status,valid_to_document_status,valid_from_match,valid_to_match
0,U001,2026-06-01,2026-11-30,2026-06-01,2026-11-30,True,True
1,U002,2026-04-01,2026-09-30,2026-04-01,2026-09-30,True,True
2,U003,2026-03-01,2026-08-31,2026-03-01,2026-08-31,True,True
3,U004,2026-05-01,2026-10-31,2026-05-01,2026-10-31,True,True
4,U005,2026-01-01,2026-06-30,2026-01-01,2026-06-30,True,True



=== 6. MVP全体チェック結果 ===
✅ alerts_integrated.csv が存在する
✅ alerts_integrated.csv に行が存在する
✅ alerts_integrated.csv に必要列が存在する
✅ action_type が設定されている
✅ alert_summary_by_user.csv が存在する
✅ alert_summary_by_user.csv に行が存在する
✅ 状態別の書類名列が存在する
✅ アラート文章が作成されている
✅ daily_records_note_reviewed.csv が存在する
✅ 対象の特記事項記録が1件存在する
✅ 特記事項AI補填案が保存されている
✅ 特記事項AI補填案が職員確認済み
✅ 特記事項AI補填案の確認者が記録されている
✅ 特記事項AI補填案の確認日時が記録されている
✅ monitoring_records_reviewed.csv が存在する
✅ 対象利用者のモニタリング記録が1件存在する
✅ 月間モニタリングAI下書きが保存されている
✅ 月間モニタリングAI下書きが職員確認済み
✅ 月間モニタリング確認者が記録されている
✅ 月間モニタリング確認日時が記録されている
✅ care_plans と document_status の有効期間が一致している

✅ Care Compliance Copilot MVP の主要フローは正常に完了しました。
✅ 統合アラート検知、状態別アラート表示、特記事項補填案、月間モニタリング下書き、職員確認済み保存まで確認済みです。


## 本MVPで検証した業務フロー

本MVPでは、訪問介護事業所における運営指導前の書類確認業務を想定し、CSVデータをもとに、未作成・未確認・未承認の書類を検知し、必要に応じてAIが文書下書き候補を提示する一連の流れを検証した。

### 1. 利用者情報・ケアプラン・日々の介護記録・書類ステータスを読み込む

利用者情報、ケアプラン、日々の介護記録、モニタリング記録、書類ステータスをCSVとして取り込み、利用者ごとの記録状況と書類状態を確認できるようにした。

### 2. 書類の未作成・未確認・未承認を検知し、状態別に対象書類名を表示する

`document_status.csv` と日々の介護記録をもとに、以下のような状態を検知する。

- 当月モニタリング記録が未確認である
- 記録内容に確認が必要な項目がある
- 特記事項が不足している
- 人による確認が必要な書類・記録が存在する

利用者別アラートでは、0〜100点などのスコアではなく、未作成・未承認・未確認／確認不足の状態別に対象書類名と件数を表示する。
また、Streamlit化を見据え、対象記録へ遷移するための `target_type`、`target_id`、`action_type` を保持する。
これにより、職員が「どの利用者の、どの書類・記録を確認すべきか」を直感的に把握できるようにした。

### 3. 必要に応じてAIが下書き候補を提示する

月間モニタリング文書や日々の介護記録の特記事項など、文章作成が必要な項目については、ケアプランと日々の介護記録を参照し、AIが下書き候補を生成する。

ただし、AIの出力は確定文書ではなく、職員が確認・修正する前提の補助案として扱う。

### 4. 職員が内容を確認し、確認済みとして保存する

AIが提示した下書きや補填案について、職員が内容を確認したうえで、確認者・確認日時・確認済みステータスを保存する。

これにより、AIが自動で書類を確定するのではなく、最終判断を人間が行う運用フローを検証した。

### 5. 本MVPで検証した価値

本MVPでは、以下の価値を検証した。

- 書類不備や未確認記録の見落としを減らせること
- 運営指導前の確認作業を効率化できること
- 対応が必要な書類名と件数を状態別に表示し、次に確認すべき対象を明確化できること
- 記録内容に基づいた文書下書きを作成できること
- AI出力を職員確認前提で扱うことで、現場運用に近い安全な設計にできること
